# MVP de Engenharia de Dados

**Aluno:** Pedro Roquim Resende Borges  
**Instituição:** PUC-Rio  
**Pós-graduação:** Ciência de Dados e Analytics  
**Disciplina:** Engenharia de Dados  
**Projeto:** Análise de avaliações de cervejas — BeerAdvocate  
**Plataforma:** Databricks

# 1. Apresentação do MVP

## BeerAdvocate Data Pipeline

Este projeto tem como objetivo construir um pipeline de Engenharia de Dados
para ingestão, tratamento, controle de qualidade, modelagem e análise de
dados de avaliações de cervejas.

### Arquitetura

Dados brutos - Bronze - Silver - Gold - Análise

### Fonte de dados

BeerAdvocate - conjunto de dados contendo avaliações de cervejas,
cervejarias, estilos, usuários e características das avaliações.

## 2. Objetivos e perguntas de negócio

O objetivo do MVP é desenvolver um pipeline de Engenharia de Dados que permita ingerir, tratar, organizar e analisar avaliações de cervejas, aplicando uma arquitetura em camadas e um modelo dimensional para apoiar consultas analíticas.

A partir dos dados disponíveis, foram definidas cinco perguntas de negócio:

1. Quais estilos de cerveja concentram o maior volume de avaliações e quais são suas notas médias?
2. Quais cervejarias apresentam o maior volume de avaliações, quantas cervejas distintas possuem na base e quais são suas notas médias?
3. Como evoluíram o volume de avaliações e as notas médias ao longo dos anos?
4. Quais cervejarias apresentam a maior diversidade de cervejas e estilos registrados no conjunto de dados?
5. Quais cervejas apresentam menor dispersão nas notas gerais, considerando um mínimo de 100 avaliações, e quais combinam consistência com notas médias elevadas?

As perguntas serão respondidas por meio de consultas SQL sobre o modelo dimensional da camada Gold, e seus resultados serão consolidados na seção 8.

# 3. Ingestão dos dados e construção da camada Bronze

Nesta etapa, será realizada a leitura do arquivo CSV original, armazenado no volume gerenciado pelo Unity Catalog. O processamento será executado com Apache Spark, permitindo examinar a estrutura dos dados, verificar a quantidade de registros e iniciar a construção da camada Bronze.

### 3.1. Leitura e validação inicial

O conjunto de dados BeerAdvocate foi disponibilizado em formato CSV e armazenado em um volume gerenciado pelo Unity Catalog, no Databricks.

A leitura foi realizada utilizando PySpark. Durante a ingestão, identificou-se que o arquivo utiliza ponto e vírgula (`;`) como delimitador, sendo necessário configurar explicitamente esse parâmetro para interpretar corretamente as 14 colunas.

Antes da persistência dos dados, são verificados o esquema, os tipos identificados automaticamente e a quantidade de registros.



In [0]:

caminho = "/Volumes/workspace/mvp_cervejas/beer_reviews_raw/beer_reviews.csv"

df_bronze = (spark.read
    .option("header", "true")
    .option("sep", ";")
    .option("inferSchema", "true")
    .csv(caminho)
)

display(df_bronze.limit(10))
  

index,brewery_id,brewery_name,review_time,review_overall,review_aroma,review_appearance,review_profilename,beer_style,review_palate,review_taste,beer_name,beer_abv,beer_beerid
0,10325,Vecchio Birraio,1234817823,1.5,2.0,2.5,stcules,Hefeweizen,1.5,1.5,Sausa Weizen,5.0,47986
1,10325,Vecchio Birraio,1235915097,3.0,2.5,3.0,stcules,English Strong Ale,3.0,3.0,Red Moon,6.2,48213
2,10325,Vecchio Birraio,1235916604,3.0,2.5,3.0,stcules,Foreign / Export Stout,3.0,3.0,Black Horse Black Beer,6.5,48215
3,10325,Vecchio Birraio,1234725145,3.0,3.0,3.5,stcules,German Pilsener,2.5,3.0,Sausa Pils,5.0,47969
4,1075,Caldera Brewing Company,1293735206,4.0,4.5,4.0,johnmichaelsen,American Double / Imperial IPA,4.0,4.5,Cauldron DIPA,7.7,64883
5,1075,Caldera Brewing Company,1325524659,3.0,3.5,3.5,oline73,Herbed / Spiced Beer,3.0,3.5,Caldera Ginger Beer,4.7,52159
6,1075,Caldera Brewing Company,1318991115,3.5,3.5,3.5,Reidrover,Herbed / Spiced Beer,4.0,4.0,Caldera Ginger Beer,4.7,52159
7,1075,Caldera Brewing Company,1306276018,3.0,2.5,3.5,alpinebryant,Herbed / Spiced Beer,2.0,3.5,Caldera Ginger Beer,4.7,52159
8,1075,Caldera Brewing Company,1290454503,4.0,3.0,3.5,LordAdmNelson,Herbed / Spiced Beer,3.5,4.0,Caldera Ginger Beer,4.7,52159
9,1075,Caldera Brewing Company,1285632924,4.5,3.5,5.0,augustgarage,Herbed / Spiced Beer,4.0,4.0,Caldera Ginger Beer,4.7,52159


### 3.2. Verificação da quantidade de registros

Após a leitura do arquivo CSV, foi realizada uma conferência
da quantidade de registros e colunas.

O conjunto apresentou **1.048.575 registros e 14 colunas**,
que serão utilizados como referência para validar a
persistência dos dados na camada Bronze.

In [0]:
print("Total de registros:", df_bronze.count())
print("Total de colunas:", len(df_bronze.columns))

Total de registros: 1048575
Total de colunas: 14


### 3.3. Persistência na camada Bronze

Os dados foram armazenados em uma tabela Delta denominada `bronze_beer_reviews`, preservando os registros e valores recebidos do CSV, sem aplicação de regras de limpeza ou exclusão de registros.

A camada Bronze constitui a primeira etapa da arquitetura Medalhão e servirá como fonte para as transformações e verificações de qualidade realizadas posteriormente na camada Silver.


In [0]:

df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.mvp_cervejas.bronze_beer_reviews")
  

### 3.4. Validação da persistência

Após a gravação da tabela Delta, foi realizada uma nova
consulta para verificar a integridade estrutural da ingestão.

A tabela Bronze apresentou **1.048.575 registros e
14 colunas**, correspondentes à quantidade identificada
na leitura inicial.

Essa conferência confirma a manutenção da quantidade de
registros e colunas durante a persistência. A qualidade
dos valores será investigada na etapa seguinte.

In [0]:

df_verificacao = spark.table(
    "workspace.mvp_cervejas.bronze_beer_reviews"
)

print("Registros na Bronze:", df_verificacao.count())
print("Colunas na Bronze:", len(df_verificacao.columns))

display(df_verificacao.limit(5))

Registros na Bronze: 1048575
Colunas na Bronze: 14


index,brewery_id,brewery_name,review_time,review_overall,review_aroma,review_appearance,review_profilename,beer_style,review_palate,review_taste,beer_name,beer_abv,beer_beerid
520947,113,Samuel Smith Old Brewery (Tadcaster),1265770749,4.5,3.5,4.0,meeekyh,English Porter,4.0,4.0,"Samuel Smith's, The Famous Taddy Porter",5.0,572
520948,113,Samuel Smith Old Brewery (Tadcaster),1265308562,4.5,4.5,5.0,Trizzle,English Porter,4.5,4.5,"Samuel Smith's, The Famous Taddy Porter",5.0,572
520949,113,Samuel Smith Old Brewery (Tadcaster),1265261622,4.0,4.5,4.0,GilGarp,English Porter,5.0,4.5,"Samuel Smith's, The Famous Taddy Porter",5.0,572
520950,113,Samuel Smith Old Brewery (Tadcaster),1265095109,2.5,3.5,4.0,Kylito87,English Porter,3.5,3.5,"Samuel Smith's, The Famous Taddy Porter",5.0,572
520951,113,Samuel Smith Old Brewery (Tadcaster),1265066114,4.5,4.0,5.0,MikeB20,English Porter,4.0,4.5,"Samuel Smith's, The Famous Taddy Porter",5.0,572


## 4. Diagnóstico de qualidade dos dados

Antes da construção da camada Silver, será realizada
uma análise da qualidade dos dados armazenados na Bronze.

O diagnóstico contemplará a identificação de valores
ausentes, possíveis duplicidades, inconsistências nas
avaliações e problemas relacionados ao teor alcoólico
(ABV) e às datas.

Os resultados orientarão as regras de tratamento e
padronização aplicadas na camada Silver.

### 4.1. Identificação de valores nulos

A primeira verificação consiste em identificar a
quantidade de valores ausentes em cada coluna do
conjunto de dados.


In [0]:

from pyspark.sql import functions as F

df_nulos = df_verificacao.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_verificacao.columns
])

display(df_nulos)
  

index,brewery_id,brewery_name,review_time,review_overall,review_aroma,review_appearance,review_profilename,beer_style,review_palate,review_taste,beer_name,beer_abv,beer_beerid
0,0,9,654,654,654,654,889,654,654,654,654,43533,817



### 4.2. Investigação dos registros incompletos

A análise identificou **817 registros sem identificador da cerveja (`beer_beerid`)**, dos quais 654 também não apresentam data de avaliação e possuem diversos outros campos ausentes.

Os outros **163 registros** possuem todos os demais campos preenchidos, exceto o teor alcoólico (`beer_abv`). Como apresentam informações válidas para as análises de avaliações, estilos e cervejarias, poderão ser preservados na camada Silver.

Também foram observadas anomalias na representação de caracteres especiais em nomes de cervejarias e cervejas, indicando a necessidade de investigar possíveis problemas de codificação.

A decisão de tratamento será documentada e aplicada na construção da camada Silver, mantendo a Bronze inalterada.
  


#### 4.2.1. Análise dos registros sem data de avaliação

Foram identificados **654 registros sem data de avaliação (`review_time`)**, que também apresentam ausência do identificador da cerveja (`beer_beerid`) e de diversos outros atributos.

Para investigar essas ocorrências, os registros foram isolados e agrupados por cervejaria. Observou-se uma concentração de registros incompletos em cervejarias cujos nomes apresentam caracteres especiais ou problemas aparentes de codificação.

A consulta a seguir apresenta uma amostra desses registros, permitindo verificar que os campos de identificação da cervejaria permanecem preenchidos, enquanto as informações necessárias para analisar as avaliações estão ausentes.

Essas ocorrências serão consideradas na definição das regras de qualidade da camada Silver. A hipótese de problemas na exportação do arquivo ainda precisa ser confirmada.
  

In [0]:

# Investigar registros com campos essenciais ausentes

df_suspeitos = df_verificacao.filter(
    F.col("review_time").isNull()
)

display(df_suspeitos.limit(20))

index,brewery_id,brewery_name,review_time,review_overall,review_aroma,review_appearance,review_profilename,beer_style,review_palate,review_taste,beer_name,beer_abv,beer_beerid
525760,9349,Mě,null,null,null,null,null,null,null,null,null,null,null
525761,9349,Mě,null,null,null,null,null,null,null,null,null,null,null
525762,9349,Mě,null,null,null,null,null,null,null,null,null,null,null
536757,19675,Abula Alus Darī,null,null,null,null,null,null,null,null,null,null,null
536758,19675,Abula Alus Darī,null,null,null,null,null,null,null,null,null,null,null
536759,19675,Abula Alus Darī,null,null,null,null,null,null,null,null,null,null,null
569188,18115,S.C. Bere Mureş,null,null,null,null,null,null,null,null,null,null,null
651279,9359,Cē,null,null,null,null,null,null,null,null,null,null,null
657750,9359,Cē,null,null,null,null,null,null,null,null,null,null,null
657751,9359,Cē,null,null,null,null,null,null,null,null,null,null,null



#### 4.2.1.1. Distribuição dos registros incompletos por cervejaria

Após a identificação dos registros sem data de avaliação, foi realizado um agrupamento por identificador e nome da cervejaria para investigar a concentração dessas ocorrências.

A análise permite identificar quais cervejarias apresentam maior quantidade de registros incompletos e observar possíveis padrões associados a problemas de codificação de caracteres especiais.

A tabela a seguir apresenta as 20 cervejarias com maior quantidade de registros nessa condição.
  

In [0]:

# Verificar o padrão dos registros incompletos

df_suspeitos = df_verificacao.filter(
    F.col("review_time").isNull()
)

df_suspeitos.select(
    "index",
    "brewery_id",
    "brewery_name",
    "review_time",
    "beer_name",
    "beer_beerid"
).groupBy(
    "brewery_id",
    "brewery_name"
).count().orderBy(
    F.desc("count")
).show(20, truncate=False)

+----------+---------------------+-----+
|brewery_id|brewery_name         |count|
+----------+---------------------+-----+
|1730      |Karlova&#267         |170  |
|580       |Browar &#321         |119  |
|576       |Zak&#322             |72   |
|4442      |PivovarskÃ½ D&#367   |62   |
|11206     |Pivovar PodkovÃ¡&#328|57   |
|16331     |Banjalu&#269         |35   |
|9359      |C&#275               |21   |
|4107      |Pivovar &#268        |19   |
|18231     |PivovarskÃ½ dv&#367  |18   |
|11205     |M&#283               |14   |
|23345     |RinkuÂški&#371       |11   |
|19610     |M&#283               |8    |
|15584     |Biha&#263            |7    |
|7223      |AS L&#257            |6    |
|13207     |Pivovar Corgo&#328   |5    |
|18229     |Pivovara Li&#269     |5    |
|11524     |Pivovar JanÃ¡&#269   |4    |
|19675     |Abula Alus Dar&#299  |3    |
|9349      |M&#283               |3    |
|20810     |Vilkmerg&#279        |3    |
+----------+---------------------+-----+
only showing top


#### 4.2.1.2. Sobreposição dos valores ausentes

Foi realizada uma análise cruzada dos campos `review_time`
e `beer_beerid` para identificar a sobreposição dos
registros com valores ausentes.

Os resultados demonstraram que os 654 registros sem data
também não possuem identificador da cerveja.

Adicionalmente, foram identificados 163 registros sem
identificador, mas com data de avaliação preenchida.

Essa distinção permite investigar separadamente os dois
grupos e definir regras de tratamento adequadas.

In [0]:

# Verificar a sobreposição dos registros incompletos

df_verificacao.groupBy(
    F.col("review_time").isNull().alias("tempo_nulo"),
    F.col("beer_beerid").isNull().alias("cerveja_nula")
).count().show()

+----------+------------+-------+
|tempo_nulo|cerveja_nula|  count|
+----------+------------+-------+
|     false|        true|    163|
|      true|        true|    654|
|     false|       false|1047758|
+----------+------------+-------+




### 4.2.2. Análise dos registros parcialmente preenchidos

Foram identificados 163 registros que apresentam data
de avaliação, mas não possuem identificador da cerveja.

A investigação demonstrou que esses registros possuem
informações de avaliação, estilo, cervejaria e nome
da cerveja, apresentando valores ausentes apenas nos
campos `beer_abv` e `beer_beerid`.

Dessa forma, esses registros poderão ser preservados
na camada Silver, pois contêm informações relevantes
para as análises propostas.

In [0]:

# Investigar registros sem identificador,
# mas com data de avaliação preenchida

df_163 = df_verificacao.filter(
    F.col("beer_beerid").isNull() &
    F.col("review_time").isNotNull()
)

display(df_163.select(
    "index",
    "brewery_name",
    "review_time",
    "review_overall",
    "beer_style",
    "beer_name",
    "beer_abv"
).limit(20))

index,brewery_name,review_time,review_overall,beer_style,beer_name,beer_abv
535606,Baltika Breweries,1320518530,3.5,Euro Dark Lager,ÂŽateckÃ½ Gus Č,null
535607,Baltika Breweries,1294342728,2.5,Euro Dark Lager,ÂŽateckÃ½ Gus Č,null
574842,Bauskas Alus,1241292794,4.0,Euro Pale Lager,GaiÂšais Bauskas Alus Senč,null
574845,Bauskas Alus,1231064320,4.0,Euro Dark Lager,TumÂšais Bauskas Alus Speciā,null
616180,Pivovar Rohozec,1250594520,2.5,Euro Dark Lager,Rohozec SkalÃ¡k Ř,null
616181,Pivovar Rohozec,1274120489,3.0,Euro Dark Lager,Rohozec SkalÃ¡k Ř,null
616182,Pivovar Rohozec,1295906895,2.0,Czech Pilsener,Rohozec SkalÃ¡k Svě,null
616183,Pivovar Rohozec,1242382091,3.0,Czech Pilsener,Rohozec SkalÃ¡k Svě,null
616184,Pivovar Rohozec,1224033566,4.5,Czech Pilsener,Rohozec SkalÃ¡k Svě,null
616185,Pivovar Rohozec,1223950145,4.5,Czech Pilsener,Rohozec SkalÃ¡k Svě,null


#### 4.2.2.1. Verificação dos campos ausentes

Para confirmar a integridade dos 163 registros parcialmente preenchidos, foi realizada uma contagem dos valores nulos em cada coluna.

Os resultados confirmaram que todos apresentam ausência de `beer_abv` e `beer_beerid`, enquanto os demais campos estão integralmente preenchidos.

Essa verificação fundamenta a decisão de preservar esses registros na camada Silver, mantendo os valores ausentes sem atribuir informações não disponíveis.

In [0]:

# Verificar os campos ausentes nos 163 registros

df_163.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_163.columns
]).display()

index,brewery_id,brewery_name,review_time,review_overall,review_aroma,review_appearance,review_profilename,beer_style,review_palate,review_taste,beer_name,beer_abv,beer_beerid
0,0,0,0,0,0,0,0,0,0,0,0,163,163


## 4.3. Análise de duplicidades

Nesta etapa, será investigada a existência de registros duplicados na camada Bronze, com o objetivo de identificar possíveis redundâncias que possam comprometer a qualidade dos dados e os resultados das análises.

Inicialmente, serão verificados os registros integralmente repetidos, desconsiderando a coluna `index`, utilizada como identificador da linha na base original. Em seguida, serão investigadas possíveis avaliações repetidas com base na combinação de usuário, cerveja e data da avaliação.

Como uma mesma cerveja pode receber diversas avaliações, a identificação de valores repetidos não implica necessariamente a existência de duplicidades indevidas. Os resultados serão analisados antes da definição das regras de tratamento da camada Silver, mantendo os dados originais preservados na Bronze.

In [0]:
 
from pyspark.sql import functions as F

df_verificacao = spark.table(
    "workspace.mvp_cervejas.bronze_beer_reviews"
)

print("Total de registros:", df_verificacao.count())

Total de registros: 1048575


### 4.3.1. Identificação de registros integralmente duplicados

Nesta etapa, será investigada a existência de registros com valores idênticos em todas as colunas, desconsiderando o campo `index`, que representa a identificação da linha na base original.

A análise permitirá identificar possíveis repetições decorrentes da origem ou da ingestão dos dados, sem confundi-las com avaliações distintas de uma mesma cerveja.

In [0]:

colunas_analise = [
    c for c in df_verificacao.columns
    if c != "index"
]

total_registros = df_verificacao.count()

total_distintos = (
    df_verificacao
    .select(*colunas_analise)
    .distinct()
    .count()
)

print("Total de registros:", total_registros)
print("Registros distintos:", total_distintos)
print(
    "Registros duplicados:",
    total_registros - total_distintos
)

Total de registros: 1048575
Registros distintos: 1047948
Registros duplicados: 627


A comparação dos registros, desconsiderando a coluna `index`, identificou **627 ocorrências excedentes de registros repetidos**, em um conjunto de 1.048.575 linhas. Esse resultado representa aproximadamente **0,06% da base**.

Antes de definir o tratamento dessas ocorrências, será investigado se as repetições estão associadas aos registros incompletos identificados anteriormente ou se também existem entre as avaliações preenchidas.

In [0]:

colunas_analise = [
    c for c in df_verificacao.columns
    if c != "index"
]

for descricao, condicao in [
    ("Registros sem data", F.col("review_time").isNull()),
    ("Registros com data", F.col("review_time").isNotNull())
]:
    df_grupo = df_verificacao.filter(condicao)

    total = df_grupo.count()
    distintos = df_grupo.select(
        *colunas_analise
    ).distinct().count()

    print(f"{descricao}:")
    print(f"  Total: {total}")
    print(f"  Repetições: {total - distintos}")

Registros sem data:
  Total: 654
  Repetições: 627
Registros com data:
  Total: 1047921
  Repetições: 0


### 4.3.2. Distribuição das duplicidades

Para compreender a origem das 627 repetições identificadas, os registros foram separados em dois grupos, conforme a presença ou ausência da data de avaliação (`review_time`).

A investigação demonstrou que **todas as 627 repetições estão concentradas nos 654 registros sem data**, os quais correspondem a apenas **27 combinações distintas de valores**, desconsiderando a coluna `index`.

Entre os **1.047.921 registros com data de avaliação, não foram identificadas duplicidades integrais**.

Esse resultado indica que as repetições estão associadas exclusivamente aos registros altamente incompletos identificados anteriormente. Dessa forma, a exclusão desses 654 registros da camada Silver poderá ser justificada pela ausência de informações essenciais às análises, sem comprometer as avaliações preenchidas. Os registros originais permanecerão preservados na Bronze.

In [0]:

df_avaliacoes_validas = df_verificacao.filter(
    F.col("review_profilename").isNotNull()
    & F.col("beer_beerid").isNotNull()
    & F.col("review_time").isNotNull()
)

df_repeticoes = (
    df_avaliacoes_validas
    .groupBy(
        "review_profilename",
        "beer_beerid",
        "review_time"
    )
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

print(
    "Combinações potencialmente repetidas:",
    df_repeticoes.count()
)

display(df_repeticoes.limit(20))

Combinações potencialmente repetidas: 4


review_profilename,beer_beerid,review_time,count
stephenlouis,12507,1062719400,2
BeerAdvocate,538,933033601,2
BeerAdvocate,556,894931201,2
david18,935,1266616822,2


### 4.3.3. Investigação de avaliações potencialmente repetidas

Após a análise de duplicidades integrais, foi realizada uma segunda verificação considerando a combinação de usuário (`review_profilename`), identificador da cerveja (`beer_beerid`) e data da avaliação (`review_time`). Foram considerados apenas os registros com esses três campos preenchidos.

Foram identificadas **4 combinações potencialmente repetidas**, cada uma com duas ocorrências. Como não foram encontradas duplicidades integrais entre os registros com data, será realizada uma comparação dos demais atributos para verificar se representam avaliações distintas ou possíveis inconsistências na base.

Nenhum registro será removido nesta etapa.

In [0]:

chaves = [
    "review_profilename",
    "beer_beerid",
    "review_time"
]

df_comparacao = (
    df_avaliacoes_validas
    .join(
        df_repeticoes.select(*chaves),
        on=chaves,
        how="inner"
    )
    .orderBy(*chaves)
)

display(df_comparacao)

review_profilename,beer_beerid,review_time,index,brewery_id,brewery_name,review_overall,review_aroma,review_appearance,beer_style,review_palate,review_taste,beer_name,beer_abv
BeerAdvocate,538,933033601,713517,9343,Beer Works,4.0,4.0,3.0,American Pale Ale (APA),4.0,4.0,Fenway Pale Ale,5.5
BeerAdvocate,538,933033601,713528,9343,Beer Works,4.5,3.5,4.0,American Pale Ale (APA),4.5,4.5,Fenway Pale Ale,5.5
BeerAdvocate,556,894931201,713173,9343,Beer Works,3.5,3.0,3.0,American Amber / Red Ale,3.0,3.0,Boston Red,5.2
BeerAdvocate,556,894931201,713142,9343,Beer Works,4.0,3.5,3.5,American Amber / Red Ale,3.5,3.5,Boston Red,5.2
david18,935,1266616822,584830,360,Warsteiner Brauerei,4.0,3.5,3.5,German Pilsener,4.0,3.5,Warsteiner Premium Verum,4.8
david18,935,1266616822,584135,360,Warsteiner Brauerei,3.0,3.5,3.5,German Pilsener,3.0,3.0,Warsteiner Premium Verum,4.8
stephenlouis,12507,1062719400,477075,4724,Gulf Islands Brewing,3.0,3.5,4.0,American Porter,3.5,4.0,Salt Spring Porter,5.5
stephenlouis,12507,1062719400,477083,4724,Gulf Islands Brewing,3.5,3.5,4.0,American Porter,4.0,3.5,Salt Spring Porter,5.5


### 4.3.4. Conclusão da análise de duplicidades
A comparação detalhada das quatro combinações potencialmente repetidas revelou diferenças nas notas atribuídas às cervejas, apesar da coincidência de usuário, identificador da cerveja e horário da avaliação.

Dessa forma, as ocorrências não constituem duplicidades integrais. Entretanto, os dados disponíveis não permitem determinar se representam avaliações independentes, alterações de avaliações anteriores ou inconsistências na base original.

Como medida conservadora, os oito registros serão preservados provisoriamente na camada Silver. A decisão evita a exclusão de avaliações potencialmente válidas e mantém documentada a limitação identificada.

## 4.4. Validação dos intervalos das avaliações

Nesta etapa, serão analisados os intervalos das notas atribuídas às cervejas, com o objetivo de identificar possíveis valores inconsistentes que possam comprometer a qualidade das análises.

Serão verificados os valores mínimos e máximos das variáveis `review_overall`, `review_aroma`, `review_appearance`, `review_palate` e `review_taste`.

Os resultados serão utilizados para avaliar a necessidade de aplicar regras de validação e tratamento na camada Silver, mantendo os registros originais preservados na Bronze.

### 4.4.1. Identificação dos valores mínimos e máximos

Para verificar a consistência das avaliações, serão calculados os valores mínimos e máximos de cada variável de pontuação. Essa análise permitirá identificar possíveis notas fora da escala esperada e orientar as regras de qualidade dos dados.

In [0]:

colunas_notas = [
    "review_overall",
    "review_aroma",
    "review_appearance",
    "review_palate",
    "review_taste"
]

df_intervalos = df_verificacao.select(
    *[
        F.min(c).alias(f"{c}_min")
        for c in colunas_notas
    ],
    *[
        F.max(c).alias(f"{c}_max")
        for c in colunas_notas
    ]
)

display(df_intervalos)

review_overall_min,review_aroma_min,review_appearance_min,review_palate_min,review_taste_min,review_overall_max,review_aroma_max,review_appearance_max,review_palate_max,review_taste_max
0.0,1.0,0.0,1.0,1.0,5.0,5.0,5.0,5.0,5.0


### 4.4.2. Investigação das avaliações com nota zero

A análise dos valores mínimos e máximos identificou notas iguais a zero nas variáveis `review_overall` e `review_appearance`, enquanto as demais variáveis apresentaram valores entre 1 e 5.

Para avaliar a extensão dessas ocorrências, será contabilizada a quantidade de registros com nota zero em cada variável e investigada a possível sobreposição entre elas. Os resultados orientarão a definição das regras de tratamento na camada Silver.

In [0]:

df_zeros = df_verificacao.select(
    F.sum(
        (F.col("review_overall") == 0).cast("int")
    ).alias("nota_geral_zero"),

    F.sum(
        (F.col("review_appearance") == 0).cast("int")
    ).alias("aparencia_zero"),

    F.sum(
        (
            (F.col("review_overall") == 0) &
            (F.col("review_appearance") == 0)
        ).cast("int")
    ).alias("ambas_zero")
)

display(df_zeros)

nota_geral_zero,aparencia_zero,ambas_zero
7,7,7


### 4.4.3. Inspeção dos registros com notas zeradas

Foram identificados **7 registros com nota zero simultaneamente nas variáveis `review_overall` e `review_appearance`**, sem ocorrências isoladas em apenas uma dessas variáveis.

Diante da pequena quantidade de ocorrências, será realizada uma inspeção individual dos registros, considerando as demais notas e informações das avaliações. O objetivo é investigar possíveis padrões e definir um tratamento adequado na camada Silver, sem alterar os dados originais da Bronze.

###4.4.4. Interpretação dos resultados
A inspeção individual demonstrou que os sete registros pertencem ao mesmo usuário (`beermut7`), sendo quatro avaliações da Red Rock Brewing Company e três da Desert Edge Brewery.

Em todos os casos, as notas geral e de aparência estão zeradas, enquanto as avaliações de aroma, paladar e sabor permanecem preenchidas com valores entre 1 e 5.

A concentração das ocorrências em um único usuário sugere uma possível particularidade no registro dessas avaliações, mas não permite determinar sua causa. Para a camada Silver, propõe-se preservar os sete registros e converter os valores zero das duas variáveis em nulos, evitando a utilização de notas potencialmente inválidas nas análises.

In [0]:

df_notas_zero = df_verificacao.filter(
    (F.col("review_overall") == 0) &
    (F.col("review_appearance") == 0)
)

display(
    df_notas_zero.select(
        "index",
        "brewery_name",
        "beer_name",
        "beer_style",
        "review_profilename",
        "review_time",
        "review_overall",
        "review_aroma",
        "review_appearance",
        "review_palate",
        "review_taste",
        "beer_abv"
    )
)

index,brewery_name,beer_name,beer_style,review_profilename,review_time,review_overall,review_aroma,review_appearance,review_palate,review_taste,beer_abv
653695,Red Rock Brewing Company,Red Rock Bavarian Weiss,Hefeweizen,beernut7,1030007859,0.0,2.0,0.0,2.5,2.0,4.0
653706,Red Rock Brewing Company,Red Rock Amber Ale,American Amber / Red Ale,beernut7,1030008142,0.0,3.0,0.0,2.5,3.5,4.0
653726,Red Rock Brewing Company,Red Rock Pilsner,German Pilsener,beernut7,1030007737,0.0,2.0,0.0,3.0,1.5,4.0
653740,Red Rock Brewing Company,Red Rock Dunkel Weizen,Dunkelweizen,beernut7,1030008039,0.0,2.0,0.0,2.5,2.0,4.0
37234,Desert Edge Brewery,Pub Pils,Czech Pilsener,beernut7,1030197670,0.0,2.0,0.0,3.0,2.0,4.0
39648,Desert Edge Brewery,Latter Days Stout,Irish Dry Stout,beernut7,1030267158,0.0,4.0,0.0,2.0,2.0,4.0
39667,Desert Edge Brewery,Utah Pale Ale,American Pale Ale (APA),beernut7,1030266889,0.0,3.0,0.0,2.0,2.0,4.0


## 4.5. Validação do teor alcoólico

Nesta etapa, será avaliada a qualidade dos dados referentes ao teor alcoólico das cervejas (`beer_abv`), considerando a presença de valores ausentes e possíveis valores extremos.

Serão analisados os valores mínimo, máximo e médio, além da distribuição dos registros com maiores graduações alcoólicas. A investigação busca distinguir características legítimas dos produtos de eventuais inconsistências, subsidiando as regras de tratamento da camada Silver.

### 4.5.1. Análise descritiva do teor alcoólico

Foram calculados os valores mínimo, máximo e médio do teor alcoólico, além da quantidade de registros com valores ausentes.

A análise identificou um teor alcoólico mínimo de **0,01%**, máximo de **57,7%** e médio de aproximadamente **7,07%**. Também foram identificados **43.533 registros sem informação de teor alcoólico**, correspondentes a aproximadamente **4,15% da base**.

Diante da amplitude observada, os valores extremos serão investigados antes da definição de regras de tratamento.

In [0]:

df_abv = df_verificacao.select(
    F.min("beer_abv").alias("abv_minimo"),
    F.max("beer_abv").alias("abv_maximo"),
    F.avg("beer_abv").alias("abv_medio"),
    F.sum(
        F.col("beer_abv").isNull().cast("int")
    ).alias("abv_nulos")
)

display(df_abv)

abv_minimo,abv_maximo,abv_medio,abv_nulos
0.01,57.7,7.066524384055585,43533


### 4.5.2. Investigação dos valores extremos

Para investigar os teores alcoólicos mais elevados, foram selecionados os 20 registros com maiores valores de `beer_abv`, juntamente com o nome da cerveja, cervejaria, estilo e avaliação geral.

A consulta identificou a **Schorschbock 57**, da cervejaria alemã Schorschbräu, classificada como Eisbock e registrada com **57,7% de álcool**. Também foram observadas avaliações da **Sink The Bismarck!**, da BrewDog, com **41% de álcool**.

Esses resultados demonstram a importância de investigar valores extremos antes de classificá-los como inconsistências.

In [0]:

display(
    df_verificacao
    .filter(F.col("beer_abv").isNotNull())
    .select(
        "brewery_name",
        "beer_name",
        "beer_style",
        "beer_abv",
        "review_overall"
    )
    .orderBy(F.desc("beer_abv"))
    .limit(20)
)

brewery_name,beer_name,beer_style,beer_abv,review_overall
SchorschbrÃ¤u,SchorschbrÃ¤u Schorschbock 57%,Eisbock,57.7,4.0
SchorschbrÃ¤u,SchorschbrÃ¤u Schorschbock 43%,Eisbock,43.0,3.5
SchorschbrÃ¤u,SchorschbrÃ¤u Schorschbock 43%,Eisbock,43.0,4.0
BrewDog,Sink The Bismarck!,American Double / Imperial IPA,41.0,3.0
BrewDog,Sink The Bismarck!,American Double / Imperial IPA,41.0,2.5
BrewDog,Sink The Bismarck!,American Double / Imperial IPA,41.0,2.0
BrewDog,Sink The Bismarck!,American Double / Imperial IPA,41.0,3.5
BrewDog,Sink The Bismarck!,American Double / Imperial IPA,41.0,5.0
BrewDog,Sink The Bismarck!,American Double / Imperial IPA,41.0,3.5
BrewDog,Sink The Bismarck!,American Double / Imperial IPA,41.0,1.0


### 4.5.3. Interpretação dos resultados

A investigação identificou que o maior teor alcoólico da base corresponde à Schorschbock 57, da cervejaria alemã Schorschbräu. O catálogo oficial da fabricante confirma a existência do produto e sua elevada graduação alcoólica, embora apresente um valor ligeiramente diferente dos 57,7% registrados na base. Dessa forma, o valor extremo será preservado na camada Silver, não sendo considerado um erro apenas por sua magnitude.

Os 43.533 valores ausentes de `beer_abv` serão investigados separadamente. Até que seja estabelecida uma justificativa para seu preenchimento, a ausência dessa informação não será utilizada como critério isolado para excluir avaliações válidas.

A análise também reforça que a qualidade dos dados deve ser avaliada independentemente da relação estatística entre teor alcoólico e notas das cervejas.

Referência: https://www.schorschbraeu.de/Schorschbock-57-Das-Staerkste-Bier-der-Welt
Acesso em: 23 set. 2026.

### 4.5.4. Investigação dos valores ausentes de teor alcoólico

A análise descritiva identificou **43.533 registros sem informação de teor alcoólico (`beer_abv`)**, correspondentes a aproximadamente **4,15% da base**.

Nesta etapa, será investigada a distribuição desses valores ausentes, verificando se estão concentrados em determinadas cervejas ou estilos. O objetivo é identificar possíveis padrões de ausência e avaliar se existem informações suficientes para justificar algum tratamento na camada Silver.

A ausência do teor alcoólico não será considerada, isoladamente, motivo para excluir avaliações válidas.

In [0]:

df_abv_ausente = df_verificacao.filter(
    F.col("beer_abv").isNull()
)

df_abv_por_cerveja = (
    df_abv_ausente
    .groupBy(
        "brewery_id",
        "brewery_name",
        "beer_beerid",
        "beer_name"
    )
    .count()
    .orderBy(F.desc("count"))
)

print(
    "Total de registros sem ABV:",
    df_abv_ausente.count()
)

display(df_abv_por_cerveja.limit(20))

Total de registros sem ABV: 43533


brewery_id,brewery_name,beer_beerid,beer_name,count
590,New Glarus Brewing Company,43456,Unplugged Berliner Weiss,241
132,Rogue Ales,362,Oregon Golden Ale,209
35,Boston Beer Company (Samuel Adams),30857,Brewer Patriot Collection - George Washington Porter,186
35,Boston Beer Company (Samuel Adams),30856,Brewer Patriot Collection - James Madison Dark Wheat Ale,177
1730,Karlovaċ,null,null,170
793,Lake Louie Brewery Company,5059,Lake Louie Warped Speed Scotch Ale,151
590,New Glarus Brewing Company,44942,Unplugged Bohemian Lager,139
14551,Green Valley Brewing Company (Crooked Creek),31029,Wild Hop Organic Amber Lager,138
3340,Dinkelacker-Schwabenbraeu AG,4032,Dinkel Acker Dark,136
3,Abita Brewing Co.,26497,Fleur-de-lis Restoration Ale,136


### 4.5.5. Verificação da disponibilidade de ABV por cerveja

A distribuição dos valores ausentes revelou concentrações de avaliações sem teor alcoólico em determinadas cervejas. Também foram identificados grupos sem informações de identificação do produto, associados aos registros incompletos investigados anteriormente.

Nesta etapa, será verificado se as cervejas que apresentam avaliações sem `beer_abv` também possuem outros registros com essa informação preenchida. A análise permitirá avaliar a possibilidade de recuperar valores a partir da própria base, sem recorrer à imputação pela média geral.

In [0]:

df_abv_disponibilidade = (
    df_verificacao
    .filter(F.col("beer_beerid").isNotNull())
    .groupBy("beer_beerid")
    .agg(
        F.sum(
            F.col("beer_abv").isNull().cast("int")
        ).alias("abv_ausentes"),

        F.count("beer_abv").alias("abv_preenchidos"),

        F.countDistinct("beer_abv").alias("abv_distintos")
    )
    .filter(F.col("abv_ausentes") > 0)
)

display(
    df_abv_disponibilidade
    .groupBy(
        F.when(
            F.col("abv_preenchidos") == 0,
            "Sem ABV disponível"
        ).when(
            F.col("abv_distintos") == 1,
            "Um único ABV disponível"
        ).otherwise(
            "Múltiplos ABVs disponíveis"
        ).alias("situacao")
    )
    .agg(
        F.count("*").alias("quantidade_cervejas"),
        F.sum("abv_ausentes").alias("registros_sem_abv")
    )
)

situacao,quantidade_cervejas,registros_sem_abv
Sem ABV disponível,10823,42716


### 4.5.6. Interpretação dos resultados e definição do tratamento
A investigação identificou **10.823 cervejas com identificador preenchido, mas sem nenhum registro de teor alcoólico disponível**, totalizando **42.716 avaliações sem ABV**.

Não foram identificadas cervejas que apresentassem simultaneamente registros com e sem teor alcoólico, o que inviabiliza a recuperação dos valores ausentes a partir de outras avaliações do mesmo produto.

Os **817 registros restantes sem ABV** correspondem às ocorrências sem identificador da cerveja, investigadas anteriormente na seção 4.2.

Dessa forma, propõe-se manter os valores ausentes de `beer_abv` como nulos na camada Silver, evitando a imputação de valores artificiais. As avaliações com os demais campos válidos serão preservadas, permitindo análises que não dependam do teor alcoólico.

## 4.6. Validação das datas das avaliações

Nesta etapa, será investigada a consistência temporal das avaliações, utilizando o campo `review_time`, originalmente armazenado em formato Unix timestamp.

Serão identificados os valores mínimo e máximo, convertidos para datas legíveis e verificada a existência de possíveis inconsistências cronológicas. Também serão considerados os registros sem data, identificados anteriormente no diagnóstico de valores ausentes.

Os resultados subsidiarão a padronização das datas e a definição das regras de tratamento na camada Silver.

### 4.6.1. Identificação do período das avaliações

Para determinar o período abrangido pela base, serão identificados os valores mínimo e máximo de `review_time` e realizada sua conversão para datas legíveis, utilizando UTC como referência.

Essa análise permitirá verificar a abrangência temporal do conjunto de dados e identificar eventuais datas incompatíveis com o histórico das avaliações.

In [0]:

from pyspark.sql import functions as F

# Padronização do fuso horário da sessão
spark.conf.set("spark.sql.session.timeZone", "UTC")

df_datas = df_verificacao.agg(
    F.min("review_time").alias("timestamp_minimo"),
    F.max("review_time").alias("timestamp_maximo"),
    F.sum(
        F.col("review_time").isNull().cast("int")
    ).alias("datas_ausentes")
)

df_datas = (
    df_datas
    .withColumn(
        "data_minima",
        F.from_unixtime("timestamp_minimo")
    )
    .withColumn(
        "data_maxima",
        F.from_unixtime("timestamp_maximo")
    )
)

display(df_datas)

timestamp_minimo,timestamp_maximo,datas_ausentes,data_minima,data_maxima
840672001,1326285348,654,1996-08-22 00:00:01,2012-01-11 12:35:48


A conversão dos timestamps identificou avaliações entre **2 de agosto de 1996** e **11 de janeiro de 2012**, utilizando UTC como referência. Foram confirmados **654 registros sem data**, já investigados na seção 4.2.

Não foram identificadas datas futuras nos limites observados. Entretanto, será analisada a distribuição anual das avaliações para investigar possíveis concentrações ou inconsistências temporais.

### 4.6.2. Distribuição das avaliações por ano

Para complementar a validação temporal, as avaliações com data preenchida serão agrupadas por ano. Essa análise permitirá observar a distribuição histórica dos registros e identificar possíveis concentrações ou anos com quantidade atípica de avaliações.

In [0]:

df_avaliacoes_ano = (
    df_verificacao
    .filter(F.col("review_time").isNotNull())
    .withColumn(
        "ano",
        F.year(F.from_unixtime("review_time").cast("timestamp"))
    )
    .groupBy("ano")
    .count()
    .orderBy("ano")
)

display(df_avaliacoes_ano)

ano,count
1996,1
1998,78
1999,53
2000,60
2001,1203
2002,14584
2003,35039
2004,43485
2005,56690
2006,82650


### 4.6.3. Investigação da avaliação mais antiga

A distribuição anual revelou uma única avaliação em 1996 e nenhuma ocorrência em 1997. A quantidade de avaliações aumenta expressivamente a partir de 2002, alcançando seu maior volume em 2011.

Como o registro de 1996 está temporalmente isolado em relação aos demais, será realizada uma inspeção individual para verificar suas informações e avaliar sua consistência. A ausência de avaliações em 1997 e a baixa quantidade nos primeiros anos não serão consideradas, isoladamente, evidências de erro.

In [0]:

display(
    df_verificacao
    .filter(
        F.year(
            F.from_unixtime("review_time").cast("timestamp")
        ) == 1996
    )
    .select(
        "index",
        "brewery_name",
        "beer_name",
        "beer_style",
        "review_profilename",
        "review_time",
        "review_overall",
        "review_aroma",
        "review_appearance",
        "review_palate",
        "review_taste"
    )
)

index,brewery_name,beer_name,beer_style,review_profilename,review_time,review_overall,review_aroma,review_appearance,review_palate,review_taste
564601,Berkshire Brewing Company Inc.,Steel Rail Extra Pale Ale,American Pale Ale (APA),Todd,840672001,4.0,3.5,3.5,4.0,4.0


### 4.6.4. Conclusão da validação temporal

A análise identificou avaliações entre **2 de agosto de 1996 e 11 de janeiro de 2012**, com maior concentração de registros entre 2008 e 2011. O ano de 2012 apresenta apenas avaliações até 11 de janeiro, não sendo diretamente comparável aos anos completos.

A inspeção do único registro de 1996 identificou uma avaliação da cerveja *Steel Rail Extra Pale Ale*, da Berkshire Brewing Company, realizada pelo usuário `Todd`. Os campos de identificação e as notas examinadas estão preenchidos e apresentam valores compatíveis com a estrutura da base. Embora não tenha sido possível confirmar independentemente a data histórica, não foram identificadas inconsistências que justifiquem sua exclusão.

Na camada Silver, os timestamps válidos serão convertidos para datas legíveis, utilizando UTC como referência. Os **654 registros sem data**, já identificados como incompletos, serão tratados conforme as regras de qualidade estabelecidas anteriormente.


## 4.7. Consolidação das regras de qualidade

Após o diagnóstico da camada Bronze, foram definidas as seguintes regras de tratamento para a construção da camada Silver. As decisões consideram a preservação das avaliações válidas, a rastreabilidade das transformações e a manutenção dos dados originais na Bronze.

| Situação identificada | Tratamento na Silver |
|---|---|
| 654 registros sem data e sem informações essenciais da avaliação | Excluir da tabela de avaliações tratadas, preservando os originais na Bronze. |
| 163 registros sem identificador da cerveja e sem ABV | Preservar as avaliações, mantendo os campos ausentes como nulos. |
| 627 repetições integrais excedentes, desconsiderando `index` | Não realizar deduplicação adicional, pois as repetições estão concentradas nos registros incompletos que serão excluídos. |
| 4 combinações repetidas de usuário, cerveja e data | Preservar os oito registros, pois a coincidência desses campos não comprova duplicidade. |
| 7 avaliações com nota geral e aparência iguais a zero | Preservar os registros e converter os valores zero dessas duas variáveis em nulos. |
| 43.533 registros sem teor alcoólico | Manter `beer_abv` nulo, sem imputação artificial. |
| Teor alcoólico máximo de 57,7% | Preservar o valor extremo, cuja plausibilidade foi investigada. |
| Datas em formato Unix timestamp | Converter para data e hora legíveis, utilizando UTC como referência. |

Essas regras serão implementadas na camada Silver e verificadas por meio de consultas de validação, garantindo que os tratamentos aplicados correspondam às decisões documentadas.

## 5. Construção da camada Silver

A camada Silver tem como objetivo disponibilizar os dados de avaliações de cervejas após a aplicação das regras de qualidade estabelecidas na seção 4.7.

Nesta etapa, serão removidos os registros sem informações essenciais para a análise, tratados os valores inconsistentes das avaliações e convertidos os timestamps Unix para data e hora em UTC. Os valores ausentes de teor alcoólico serão preservados, assim como as avaliações potencialmente repetidas cuja duplicidade não foi comprovada.

As transformações serão realizadas a partir da tabela Bronze, mantendo os dados originais inalterados. O resultado será persistido em formato Delta no Unity Catalog.

### 5.1. Aplicação das regras de tratamento

In [0]:

from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")

df_silver = (
    spark.table("workspace.mvp_cervejas.bronze_beer_reviews")

    # Remove registros sem informações essenciais da avaliação
    .filter(F.col("review_time").isNotNull())

    # Trata as notas zeradas identificadas no diagnóstico
    .withColumn(
        "review_overall",
        F.when(F.col("review_overall") == 0, None)
         .otherwise(F.col("review_overall"))
    )
    .withColumn(
        "review_appearance",
        F.when(F.col("review_appearance") == 0, None)
         .otherwise(F.col("review_appearance"))
    )

    # Converte o timestamp Unix para data e hora
    .withColumn(
        "review_datetime",
        F.from_unixtime("review_time").cast("timestamp")
    )
    .withColumn(
        "review_date",
        F.to_date("review_datetime")
    )
)

print("Registros na Silver:", df_silver.count())
display(df_silver.limit(5))

Registros na Silver: 1047921


index,brewery_id,brewery_name,review_time,review_overall,review_aroma,review_appearance,review_profilename,beer_style,review_palate,review_taste,beer_name,beer_abv,beer_beerid,review_datetime,review_date
520947,113,Samuel Smith Old Brewery (Tadcaster),1265770749,4.5,3.5,4.0,meeekyh,English Porter,4.0,4.0,"Samuel Smith's, The Famous Taddy Porter",5.0,572,2010-02-10T02:59:09.000Z,2010-02-10
520948,113,Samuel Smith Old Brewery (Tadcaster),1265308562,4.5,4.5,5.0,Trizzle,English Porter,4.5,4.5,"Samuel Smith's, The Famous Taddy Porter",5.0,572,2010-02-04T18:36:02.000Z,2010-02-04
520949,113,Samuel Smith Old Brewery (Tadcaster),1265261622,4.0,4.5,4.0,GilGarp,English Porter,5.0,4.5,"Samuel Smith's, The Famous Taddy Porter",5.0,572,2010-02-04T05:33:42.000Z,2010-02-04
520950,113,Samuel Smith Old Brewery (Tadcaster),1265095109,2.5,3.5,4.0,Kylito87,English Porter,3.5,3.5,"Samuel Smith's, The Famous Taddy Porter",5.0,572,2010-02-02T07:18:29.000Z,2010-02-02
520951,113,Samuel Smith Old Brewery (Tadcaster),1265066114,4.5,4.0,5.0,MikeB20,English Porter,4.0,4.5,"Samuel Smith's, The Famous Taddy Porter",5.0,572,2010-02-01T23:15:14.000Z,2010-02-01


### 5.2. Validação das transformações

Após a aplicação das regras de tratamento, a camada Silver passou a conter **1.047.921 registros**, resultado da exclusão dos 654 registros incompletos identificados no diagnóstico da Bronze.

Nesta etapa, serão verificadas a conversão das notas zeradas em valores nulos, a preservação dos registros com teor alcoólico ausente e a criação das colunas de data. O objetivo é confirmar que as transformações foram aplicadas conforme as regras documentadas na seção 4.7, antes da persistência da tabela Delta.

In [0]:

df_validacao_silver = df_silver.agg(
    F.count("*").alias("total_registros"),

    F.sum(
        F.col("review_overall").isNull().cast("int")
    ).alias("nota_geral_nula"),

    F.sum(
        F.col("review_appearance").isNull().cast("int")
    ).alias("aparencia_nula"),

    F.sum(
        F.col("beer_abv").isNull().cast("int")
    ).alias("abv_nulo"),

    F.sum(
        F.col("review_datetime").isNull().cast("int")
    ).alias("data_nula")
)

display(df_validacao_silver)

total_registros,nota_geral_nula,aparencia_nula,abv_nulo,data_nula
1047921,7,7,42879,0


### 5.3. Persistência da camada Silver

Após a validação das transformações, os dados tratados serão persistidos em uma tabela Delta gerenciada pelo Unity Catalog, denominada `workspace.mvp_cervejas.silver_beer_reviews`.

A utilização do formato Delta permite armazenar os dados tratados de maneira estruturada e disponibilizá-los para a construção da camada Gold. A tabela Bronze permanecerá inalterada, preservando os registros originais e a rastreabilidade do processo.

In [0]:

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.mvp_cervejas.silver_beer_reviews")

print("Tabela Silver salva com sucesso!")

Tabela Silver salva com sucesso!


### 5.4. Validação da persistência

Para confirmar a gravação da camada Silver, a tabela Delta será novamente carregada a partir do Unity Catalog. Serão verificadas a quantidade de registros, a estrutura das colunas e a manutenção das regras de qualidade após a persistência.

Essa validação garante que a próxima etapa do pipeline utilize os dados efetivamente armazenados, e não apenas o DataFrame temporário criado durante as transformações.

In [0]:

df_silver_salva = spark.table(
    "workspace.mvp_cervejas.silver_beer_reviews"
)

print("Registros:", df_silver_salva.count())
print("Colunas:", len(df_silver_salva.columns))

display(df_silver_salva.limit(5))

Registros: 1047921
Colunas: 16


index,brewery_id,brewery_name,review_time,review_overall,review_aroma,review_appearance,review_profilename,beer_style,review_palate,review_taste,beer_name,beer_abv,beer_beerid,review_datetime,review_date
0,10325,Vecchio Birraio,1234817823,1.5,2.0,2.5,stcules,Hefeweizen,1.5,1.5,Sausa Weizen,5.0,47986,2009-02-16T20:57:03.000Z,2009-02-16
1,10325,Vecchio Birraio,1235915097,3.0,2.5,3.0,stcules,English Strong Ale,3.0,3.0,Red Moon,6.2,48213,2009-03-01T13:44:57.000Z,2009-03-01
2,10325,Vecchio Birraio,1235916604,3.0,2.5,3.0,stcules,Foreign / Export Stout,3.0,3.0,Black Horse Black Beer,6.5,48215,2009-03-01T14:10:04.000Z,2009-03-01
3,10325,Vecchio Birraio,1234725145,3.0,3.0,3.5,stcules,German Pilsener,2.5,3.0,Sausa Pils,5.0,47969,2009-02-15T19:12:25.000Z,2009-02-15
4,1075,Caldera Brewing Company,1293735206,4.0,4.5,4.0,johnmichaelsen,American Double / Imperial IPA,4.0,4.5,Cauldron DIPA,7.7,64883,2010-12-30T18:53:26.000Z,2010-12-30


## 6. Modelagem e construção da camada Gold

A camada Gold será estruturada por meio de um modelo dimensional em esquema estrela, visando organizar os dados tratados da Silver para consultas analíticas e responder às perguntas de negócio do MVP.

O modelo proposto contém uma tabela fato de avaliações e três dimensões: cerveja, cervejaria e tempo. A tabela fato preservará a granularidade de uma linha por avaliação, enquanto as dimensões disponibilizarão os atributos necessários para agrupamentos e análises.

### 6.1. Definição e validação do modelo dimensional

Antes da construção das tabelas, serão verificadas a unicidade dos identificadores de cervejas e cervejarias e a consistência dos atributos associados a cada identificador.

Essa validação permitirá identificar possíveis conflitos cadastrais e definir regras de modelagem que evitem relacionamentos ambíguos ou multiplicação indevida de registros nas consultas analíticas.

###6.1.1. Verificação das chaves das dimensões

Antes de criar as dimensões, precisamos saber se cada beer_beerid corresponde a um único nome, estilo e cervejaria, e se cada brewery_id corresponde a um único nome de cervejaria.

In [0]:

from pyspark.sql import functions as F

df_silver_salva = spark.table(
    "workspace.mvp_cervejas.silver_beer_reviews"
)

# Verifica conflitos de atributos para um mesmo ID de cerveja
df_conflitos_cerveja = (
    df_silver_salva
    .filter(F.col("beer_beerid").isNotNull())
    .groupBy("beer_beerid")
    .agg(
        F.countDistinct("beer_name").alias("nomes"),
        F.countDistinct("beer_style").alias("estilos"),
        F.countDistinct("brewery_id").alias("cervejarias"),
        F.countDistinct("beer_abv").alias("teores_alcoolicos")
    )
    .filter(
        (F.col("nomes") > 1) |
        (F.col("estilos") > 1) |
        (F.col("cervejarias") > 1) |
        (F.col("teores_alcoolicos") > 1)
    )
)

# Verifica conflitos de nomes para um mesmo ID de cervejaria
df_conflitos_cervejaria = (
    df_silver_salva
    .groupBy("brewery_id")
    .agg(
        F.countDistinct("brewery_name").alias("nomes")
    )
    .filter(F.col("nomes") > 1)
)

print(
    "IDs de cerveja com atributos conflitantes:",
    df_conflitos_cerveja.count()
)

print(
    "IDs de cervejaria com nomes conflitantes:",
    df_conflitos_cervejaria.count()
)

display(df_conflitos_cerveja.limit(20))
display(df_conflitos_cervejaria.limit(20))

IDs de cerveja com atributos conflitantes: 0
IDs de cervejaria com nomes conflitantes: 0


beer_beerid,nomes,estilos,cervejarias,teores_alcoolicos


brewery_id,nomes


###Interpretação dos Resultados
A verificação não identificou conflitos entre os atributos associados aos identificadores de cervejas e cervejarias. Para os registros com identificadores preenchidos, não foram encontrados múltiplos nomes, estilos, cervejarias ou teores alcoólicos não nulos associados ao mesmo identificador de cerveja, nem múltiplos nomes associados ao mesmo identificador de cervejaria.

Esse resultado permite utilizar os identificadores originais como chaves das respectivas dimensões, desde que a unicidade e a integridade dos registros sejam verificadas após sua construção. Os registros com identificadores ausentes serão tratados separadamente, conforme as regras estabelecidas na camada Silver.

### 6.2. Construção das dimensões da camada Gold

A partir dos dados tratados da Silver, serão construídas as dimensões de cervejaria, cerveja e tempo, que fornecerão os atributos descritivos necessários às análises.

Cada dimensão será estruturada de modo a apresentar uma única linha por chave, evitando a multiplicação indevida de avaliações nos relacionamentos com a tabela fato. As tabelas serão persistidas em formato Delta no Unity Catalog.

###6.2.1. Dimensão de cervejarias

Esta dimensão terá uma linha por cervejaria identificada, contendo seu código e nome. Vamos preservar o nome disponível na Silver e verificar se há identificadores ou nomes ausentes.

In [0]:

df_dim_cervejaria = (
    df_silver_salva
    .filter(F.col("brewery_id").isNotNull())
    .groupBy("brewery_id")
    .agg(
        F.first("brewery_name", ignorenulls=True)
         .alias("brewery_name")
    )
)

print("Cervejarias:", df_dim_cervejaria.count())

print(
    "IDs duplicados:",
    df_dim_cervejaria.count()
    - df_dim_cervejaria.select("brewery_id").distinct().count()
)

print(
    "Nomes ausentes:",
    df_dim_cervejaria
    .filter(F.col("brewery_name").isNull())
    .count()
)

display(df_dim_cervejaria.orderBy("brewery_id").limit(10))

Cervejarias: 3817
IDs duplicados: 0
Nomes ausentes: 1


brewery_id,brewery_name
1,"Plzensky Prazdroj, a. s."
2,Yellow Rose Brewing Company
3,Abita Brewing Co.
5,Yakima Brewing Co. / Bert Grant's Ales
8,Widmer Brothers Brewing Company
10,Whim Ales
13,Commonwealth Fish & Beer Company
14,Cambridge Brewing Company
21,Union Export Brewery
22,Unibroue


###6.2.1.1 Investigação do nome ausente

In [0]:

display(
    df_dim_cervejaria
    .filter(F.col("brewery_name").isNull())
)


brewery_id,brewery_name
1193,null


A investigação identificou que a cervejaria de código `1193` não possui nome disponível na camada Silver. Como não foi encontrada informação suficiente para identificar a cervejaria, o registro será preservado na dimensão com o campo `brewery_name` nulo.

A dimensão contém **3.817 cervejarias**, sem duplicidade de identificadores e com apenas um nome ausente. A manutenção desse registro preserva a integridade dos relacionamentos com a tabela fato e documenta uma limitação de qualidade dos dados de origem.

#### 6.2.1.2. Persistência e validação da dimensão de cervejarias

Após a verificação da unicidade dos identificadores e a investigação do nome ausente, a dimensão de cervejarias foi persistida em formato Delta no Unity Catalog, na tabela `workspace.mvp_cervejas.gold_dim_cervejaria`.

Para validar a persistência, a tabela foi novamente carregada e sua quantidade de registros foi conferida. A dimensão armazenada contém **3.817 cervejarias**, correspondendo à quantidade identificada durante sua construção.

A tabela possui uma linha por `brewery_id` e preserva o nome ausente da cervejaria de código `1193`, conforme a decisão de tratamento documentada anteriormente.

In [0]:

df_dim_cervejaria.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.mvp_cervejas.gold_dim_cervejaria"
    )

# Validação da persistência
df_dim_cervejaria_salva = spark.table(
    "workspace.mvp_cervejas.gold_dim_cervejaria"
)

print("Cervejarias:", df_dim_cervejaria_salva.count())

display(df_dim_cervejaria_salva.limit(10))

Cervejarias: 3817


brewery_id,brewery_name
8802,Conwy Brewery Ltd
14756,Bluegrass Brewing Co.
7314,Stadtbrauerei Arnstadt
22167,Fulton Beer
7402,Wasatch Brew Pub
60,Cooperstown Brewing Company
860,The Vermont Pub & Brewery
17285,Mattingly Brewing Company
24892,Brewfist
2098,Terminal Gravity Brewing Company


### 6.2.2. Dimensão de cervejas

A dimensão de cervejas será construída a partir dos registros tratados da camada Silver, utilizando `beer_beerid` como chave e reunindo os atributos descritivos de cada produto: nome, estilo, teor alcoólico e identificador da cervejaria.

Os registros sem identificador da cerveja não serão utilizados na construção desta dimensão, mas permanecerão disponíveis na tabela fato de avaliações. Os valores ausentes de `beer_abv` serão preservados como nulos, conforme as regras de qualidade definidas anteriormente.

A consolidação será realizada por identificador, recuperando os atributos não nulos disponíveis. Após a construção, serão verificadas a quantidade de cervejas, a unicidade das chaves e a ocorrência de atributos ausentes.

In [0]:

from pyspark.sql import functions as F

df_dim_cerveja = (
    df_silver_salva
    .filter(F.col("beer_beerid").isNotNull())
    .groupBy("beer_beerid")
    .agg(
        F.first("beer_name", ignorenulls=True)
         .alias("beer_name"),

        F.first("beer_style", ignorenulls=True)
         .alias("beer_style"),

        F.first("beer_abv", ignorenulls=True)
         .alias("beer_abv"),

        F.first("brewery_id", ignorenulls=True)
         .alias("brewery_id")
    )
)

# Validação da dimensão
total_cervejas = df_dim_cerveja.count()

ids_distintos = (
    df_dim_cerveja
    .select("beer_beerid")
    .distinct()
    .count()
)

print("Cervejas identificadas:", total_cervejas)
print("IDs duplicados:", total_cervejas - ids_distintos)

# Verificação de atributos ausentes
df_nulos_cerveja = df_dim_cerveja.agg(
    F.sum(F.col("beer_name").isNull().cast("int"))
     .alias("nomes_ausentes"),

    F.sum(F.col("beer_style").isNull().cast("int"))
     .alias("estilos_ausentes"),

    F.sum(F.col("beer_abv").isNull().cast("int"))
     .alias("abv_ausentes"),

    F.sum(F.col("brewery_id").isNull().cast("int"))
     .alias("cervejarias_ausentes")
)

display(df_nulos_cerveja)

display(
    df_dim_cerveja
    .orderBy("beer_beerid")
    .limit(10)
)

Cervejas identificadas: 42551
IDs duplicados: 0


nomes_ausentes,estilos_ausentes,abv_ausentes,cervejarias_ausentes
0,0,10823,0


beer_beerid,beer_name,beer_style,beer_abv,brewery_id
3,Cactus Queen IPA,American IPA,null,2
4,Wildcatter's Crude Stout,American Stout,null,2
5,Amber,Vienna Lager,4.5,3
6,Turbodog,English Brown Ale,5.6,3
7,Purple Haze,Fruit / Vegetable Beer,4.2,3
8,Wheat,American Adjunct Lager,4.2,3
9,Golden,American Pale Lager,4.2,3
11,Bert Grant's Scottish Ale,Scottish Ale,4.7,5
12,Bert Grant's Imperial Stout,Russian Imperial Stout,6.0,5
13,Bert Grant's IPA,English India Pale Ale (IPA),4.2,5


#### 6.2.2.1. Interpretação dos resultados

A construção da dimensão identificou **42.551 cervejas distintas**, sem duplicidade de identificadores. Todos os produtos possuem nome, estilo e identificador de cervejaria preenchidos.

Foram identificadas **10.823 cervejas sem informação de teor alcoólico**, resultado consistente com a investigação realizada na seção 4.5. Como não foram encontrados valores disponíveis para recuperar esse atributo a partir de outras avaliações do mesmo produto, os valores ausentes serão preservados como nulos.

Os 163 registros de avaliações sem identificador da cerveja não integram esta dimensão, mas permanecerão disponíveis na tabela fato, conforme as regras de tratamento estabelecidas para a camada Gold.

#### 6.2.2.2. Persistência e validação da dimensão de cervejas

Após a consolidação dos atributos e a verificação da unicidade dos identificadores, a dimensão de cervejas será persistida em formato Delta no Unity Catalog, na tabela `workspace.mvp_cervejas.gold_dim_cerveja`.

A validação será realizada por meio da leitura da tabela persistida e da conferência da quantidade de registros e colunas. A dimensão deverá conter **42.551 cervejas e cinco colunas**, preservando os valores ausentes de teor alcoólico.

In [0]:

# Persistência da dimensão de cervejas
df_dim_cerveja.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.mvp_cervejas.gold_dim_cerveja")

# Leitura da tabela persistida
df_dim_cerveja_salva = spark.table(
    "workspace.mvp_cervejas.gold_dim_cerveja"
)

print("Cervejas:", df_dim_cerveja_salva.count())
print("Colunas:", len(df_dim_cerveja_salva.columns))

display(
    df_dim_cerveja_salva
    .orderBy("beer_beerid")
    .limit(10)
)

Cervejas: 42551
Colunas: 5


beer_beerid,beer_name,beer_style,beer_abv,brewery_id
3,Cactus Queen IPA,American IPA,null,2
4,Wildcatter's Crude Stout,American Stout,null,2
5,Amber,Vienna Lager,4.5,3
6,Turbodog,English Brown Ale,5.6,3
7,Purple Haze,Fruit / Vegetable Beer,4.2,3
8,Wheat,American Adjunct Lager,4.2,3
9,Golden,American Pale Lager,4.2,3
11,Bert Grant's Scottish Ale,Scottish Ale,4.7,5
12,Bert Grant's Imperial Stout,Russian Imperial Stout,6.0,5
13,Bert Grant's IPA,English India Pale Ale (IPA),4.2,5


### 6.2.3. Dimensão de tempo

A dimensão de tempo será construída a partir das datas padronizadas na camada Silver, contendo uma linha por data distinta de avaliação.

Serão derivados os atributos de ano, mês, trimestre e dia, permitindo análises temporais e agrupamentos por diferentes períodos. A coluna `review_date` será utilizada como chave da dimensão.

Após sua construção, serão verificadas a quantidade de datas distintas, a unicidade da chave e a ausência de datas nulas, antes da persistência em formato Delta.

In [0]:

from pyspark.sql import functions as F

df_dim_tempo = (
    df_silver_salva
    .select("review_date")
    .filter(F.col("review_date").isNotNull())
    .distinct()
    .withColumn("ano", F.year("review_date"))
    .withColumn("mes", F.month("review_date"))
    .withColumn("trimestre", F.quarter("review_date"))
    .withColumn("dia", F.dayofmonth("review_date"))
)

# Validação da dimensão
total_datas = df_dim_tempo.count()

datas_distintas = (
    df_dim_tempo
    .select("review_date")
    .distinct()
    .count()
)

print("Datas identificadas:", total_datas)
print("Datas duplicadas:", total_datas - datas_distintas)

print(
    "Datas nulas:",
    df_dim_tempo
    .filter(F.col("review_date").isNull())
    .count()
)

display(
    df_dim_tempo
    .orderBy("review_date")
    .limit(10)
)

Datas identificadas: 4010
Datas duplicadas: 0
Datas nulas: 0


review_date,ano,mes,trimestre,dia
1996-08-22,1996,8,3,22
1998-01-10,1998,1,1,10
1998-01-13,1998,1,1,13
1998-01-22,1998,1,1,22
1998-01-28,1998,1,1,28
1998-02-06,1998,2,1,6
1998-02-14,1998,2,1,14
1998-02-17,1998,2,1,17
1998-02-18,1998,2,1,18
1998-02-19,1998,2,1,19


#### 6.2.3.1. Interpretação dos resultados

A construção da dimensão de tempo identificou **4.010 datas distintas**, sem duplicidade de chaves ou valores nulos. Os atributos de ano, mês, trimestre e dia foram derivados da coluna `review_date`, previamente padronizada na camada Silver.

A dimensão contempla as datas efetivamente presentes nas avaliações, abrangendo o período de 1996 a 2012. Portanto, não constitui um calendário completo, mas permite realizar agrupamentos e análises temporais dos registros disponíveis.

#### 6.2.3.2. Persistência e validação da dimensão de tempo

Após a validação da unicidade das datas e dos atributos temporais, a dimensão será persistida em formato Delta no Unity Catalog, na tabela `workspace.mvp_cervejas.gold_dim_tempo`.

A persistência será verificada por meio da leitura da tabela armazenada e da conferência de sua quantidade de registros e colunas. O resultado esperado é de **4.010 datas e cinco colunas**.

In [0]:

# Persistência da dimensão de tempo
df_dim_tempo.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.mvp_cervejas.gold_dim_tempo")

# Validação da tabela persistida
df_dim_tempo_salva = spark.table(
    "workspace.mvp_cervejas.gold_dim_tempo"
)

print("Datas:", df_dim_tempo_salva.count())
print("Colunas:", len(df_dim_tempo_salva.columns))

display(
    df_dim_tempo_salva
    .orderBy("review_date")
    .limit(10)
)

Datas: 4010
Colunas: 5


review_date,ano,mes,trimestre,dia
1996-08-22,1996,8,3,22
1998-01-10,1998,1,1,10
1998-01-13,1998,1,1,13
1998-01-22,1998,1,1,22
1998-01-28,1998,1,1,28
1998-02-06,1998,2,1,6
1998-02-14,1998,2,1,14
1998-02-17,1998,2,1,17
1998-02-18,1998,2,1,18
1998-02-19,1998,2,1,19


## 6.3. Construção da tabela fato de avaliações

A tabela fato será construída a partir da camada Silver, mantendo a granularidade de **uma linha por avaliação**. Serão preservadas as notas atribuídas pelos usuários, a identificação do avaliador e as chaves necessárias para relacionar cada avaliação às dimensões de cerveja, cervejaria e tempo.

### 6.3.1. Definição da granularidade e dos atributos

A coluna `index`, proveniente da base original, será utilizada como identificador técnico da avaliação. As colunas `beer_beerid`, `brewery_id` e `review_date` permitirão os relacionamentos com as respectivas dimensões.

Os registros sem identificador da cerveja serão preservados com chave nula, conforme as regras de qualidade estabelecidas anteriormente. A tabela fato não armazenará novamente os atributos descritivos das dimensões, evitando redundância no modelo analítico.

In [0]:

from pyspark.sql import functions as F

df_fato_avaliacoes = (
    df_silver_salva
    .select(
        F.col("index").alias("avaliacao_id"),
        "beer_beerid",
        "brewery_id",
        "review_date",
        "review_datetime",
        "review_profilename",
        "review_overall",
        "review_aroma",
        "review_appearance",
        "review_palate",
        "review_taste"
    )
)

# Verificação da granularidade
total_avaliacoes = df_fato_avaliacoes.count()

ids_distintos = (
    df_fato_avaliacoes
    .select("avaliacao_id")
    .distinct()
    .count()
)

print("Total de avaliações:", total_avaliacoes)
print("IDs duplicados:", total_avaliacoes - ids_distintos)

# Verificação das chaves
df_chaves_fato = df_fato_avaliacoes.agg(
    F.sum(F.col("avaliacao_id").isNull().cast("int"))
     .alias("avaliacoes_sem_id"),

    F.sum(F.col("beer_beerid").isNull().cast("int"))
     .alias("cervejas_sem_id"),

    F.sum(F.col("brewery_id").isNull().cast("int"))
     .alias("cervejarias_sem_id"),

    F.sum(F.col("review_date").isNull().cast("int"))
     .alias("datas_ausentes")
)

display(df_chaves_fato)
display(df_fato_avaliacoes.limit(10))

Total de avaliações: 1047921
IDs duplicados: 0


avaliacoes_sem_id,cervejas_sem_id,cervejarias_sem_id,datas_ausentes
0,163,0,0


avaliacao_id,beer_beerid,brewery_id,review_date,review_datetime,review_profilename,review_overall,review_aroma,review_appearance,review_palate,review_taste
0,47986,10325,2009-02-16,2009-02-16T20:57:03.000Z,stcules,1.5,2.0,2.5,1.5,1.5
1,48213,10325,2009-03-01,2009-03-01T13:44:57.000Z,stcules,3.0,2.5,3.0,3.0,3.0
2,48215,10325,2009-03-01,2009-03-01T14:10:04.000Z,stcules,3.0,2.5,3.0,3.0,3.0
3,47969,10325,2009-02-15,2009-02-15T19:12:25.000Z,stcules,3.0,3.0,3.5,2.5,3.0
4,64883,1075,2010-12-30,2010-12-30T18:53:26.000Z,johnmichaelsen,4.0,4.5,4.0,4.0,4.5
5,52159,1075,2012-01-02,2012-01-02T17:17:39.000Z,oline73,3.0,3.5,3.5,3.0,3.5
6,52159,1075,2011-10-19,2011-10-19T02:25:15.000Z,Reidrover,3.5,3.5,3.5,4.0,4.0
7,52159,1075,2011-05-24,2011-05-24T22:26:58.000Z,alpinebryant,3.0,2.5,3.5,2.0,3.5
8,52159,1075,2010-11-22,2010-11-22T19:35:03.000Z,LordAdmNelson,4.0,3.0,3.5,3.5,4.0
9,52159,1075,2010-09-28,2010-09-28T00:15:24.000Z,augustgarage,4.5,3.5,5.0,4.0,4.0


### 6.3.3. Validação da integridade referencial

A construção inicial da tabela fato resultou em **1.047.921 avaliações**, sem duplicidade ou ausência de identificadores técnicos. Foram preservadas **163 avaliações sem identificador da cerveja**, enquanto todos os registros apresentam identificador da cervejaria e data de avaliação.

Antes da persistência, será verificada a integridade referencial entre a tabela fato e as três dimensões do modelo. A validação permitirá identificar eventuais chaves preenchidas que não encontrem correspondência nas dimensões de cerveja, cervejaria e tempo.

Os identificadores de cerveja ausentes serão analisados separadamente, pois sua preservação constitui uma decisão documentada de qualidade dos dados.

In [0]:

# Verifica chaves preenchidas sem correspondência
# nas respectivas dimensões.

df_sem_cerveja = (
    df_fato_avaliacoes
    .filter(F.col("beer_beerid").isNotNull())
    .join(
        df_dim_cerveja_salva.select("beer_beerid"),
        on="beer_beerid",
        how="left_anti"
    )
)

df_sem_cervejaria = (
    df_fato_avaliacoes
    .join(
        df_dim_cervejaria_salva.select("brewery_id"),
        on="brewery_id",
        how="left_anti"
    )
)

df_sem_tempo = (
    df_fato_avaliacoes
    .join(
        df_dim_tempo_salva.select("review_date"),
        on="review_date",
        how="left_anti"
    )
)

print(
    "Chaves de cerveja sem correspondência:",
    df_sem_cerveja.count()
)

print(
    "Chaves de cervejaria sem correspondência:",
    df_sem_cervejaria.count()
)

print(
    "Datas sem correspondência:",
    df_sem_tempo.count()
)

Chaves de cerveja sem correspondência: 0
Chaves de cervejaria sem correspondência: 0
Datas sem correspondência: 0


### 6.3.4. Persistência e validação da tabela fato

A validação da integridade referencial não identificou chaves preenchidas sem correspondência nas dimensões de cerveja, cervejaria e tempo. As **163 avaliações sem identificador da cerveja** foram preservadas, conforme as regras de qualidade definidas na camada Silver.

Após essas verificações, a tabela fato será persistida em formato Delta no Unity Catalog, com o nome `workspace.mvp_cervejas.gold_fato_avaliacoes`.

A persistência será validada por meio da leitura da tabela armazenada e da conferência da quantidade de registros e colunas, verificando a manutenção da granularidade de uma linha por avaliação.

In [0]:

# Persistência da tabela fato em Delta
df_fato_avaliacoes.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.mvp_cervejas.gold_fato_avaliacoes")

# Leitura e validação da tabela persistida
df_fato_salva = spark.table(
    "workspace.mvp_cervejas.gold_fato_avaliacoes"
)

print("Avaliações:", df_fato_salva.count())
print("Colunas:", len(df_fato_salva.columns))

display(
    df_fato_salva
    .orderBy("avaliacao_id")
    .limit(10)
)

Avaliações: 1047921
Colunas: 11


avaliacao_id,beer_beerid,brewery_id,review_date,review_datetime,review_profilename,review_overall,review_aroma,review_appearance,review_palate,review_taste
0,47986,10325,2009-02-16,2009-02-16T20:57:03.000Z,stcules,1.5,2.0,2.5,1.5,1.5
1,48213,10325,2009-03-01,2009-03-01T13:44:57.000Z,stcules,3.0,2.5,3.0,3.0,3.0
2,48215,10325,2009-03-01,2009-03-01T14:10:04.000Z,stcules,3.0,2.5,3.0,3.0,3.0
3,47969,10325,2009-02-15,2009-02-15T19:12:25.000Z,stcules,3.0,3.0,3.5,2.5,3.0
4,64883,1075,2010-12-30,2010-12-30T18:53:26.000Z,johnmichaelsen,4.0,4.5,4.0,4.0,4.5
5,52159,1075,2012-01-02,2012-01-02T17:17:39.000Z,oline73,3.0,3.5,3.5,3.0,3.5
6,52159,1075,2011-10-19,2011-10-19T02:25:15.000Z,Reidrover,3.5,3.5,3.5,4.0,4.0
7,52159,1075,2011-05-24,2011-05-24T22:26:58.000Z,alpinebryant,3.0,2.5,3.5,2.0,3.5
8,52159,1075,2010-11-22,2010-11-22T19:35:03.000Z,LordAdmNelson,4.0,3.0,3.5,3.5,4.0
9,52159,1075,2010-09-28,2010-09-28T00:15:24.000Z,augustgarage,4.5,3.5,5.0,4.0,4.0


## 6.4. Catálogo de Dados

O catálogo de dados documenta as estruturas utilizadas no MVP, descrevendo a finalidade de cada tabela, seus atributos, tipos de dados, domínios de valores e origem das informações.

As tabelas foram organizadas no Unity Catalog, no catálogo `workspace` e no schema `mvp_cervejas`, seguindo a arquitetura Medalhão. A camada Bronze preserva os registros ingeridos, a Silver contém os dados tratados e a Gold disponibiliza o modelo dimensional utilizado nas consultas analíticas.

Para documentar a linhagem dos dados, serão identificadas as relações entre o arquivo CSV original e as tabelas resultantes das transformações, incluindo as chaves utilizadas na construção das dimensões e da tabela fato.

### 6.4.1. Camada Bronze — Dicionário de dados

**Tabela:** `workspace.mvp_cervejas.bronze_beer_reviews`  
**Camada:** Bronze  
**Granularidade:** uma linha por registro ingerido do arquivo original.  
**Origem:** arquivo `beer_reviews.csv`, armazenado no volume `beer_reviews_raw`.

A camada Bronze preserva os 1.048.575 registros ingeridos do arquivo CSV, mantendo os atributos e valores da fonte original para permitir o diagnóstico da qualidade e o reprocessamento das etapas posteriores.

| Coluna | Tipo | Descrição | Domínio e regras |
|---|---|---|---|
| `index` | INT | Índice do registro no arquivo de origem. | Identificador original, preservado sem tratamento. |
| `brewery_id` | INT | Identificador da cervejaria. | Código numérico da fonte. |
| `brewery_name` | STRING | Nome da cervejaria. | Texto original; pode conter valores ausentes. |
| `review_time` | INT | Data e horário da avaliação em timestamp Unix. | Valor original, inclusive quando ausente. |
| `review_overall` | DOUBLE | Nota geral da cerveja. | Valor original, inclusive notas inválidas. |
| `review_aroma` | DOUBLE | Nota atribuída ao aroma. | Escala de avaliação da fonte. |
| `review_appearance` | DOUBLE | Nota atribuída à aparência. | Valor original, inclusive notas inválidas. |
| `review_profilename` | STRING | Nome de perfil do autor da avaliação. | Texto original. |
| `beer_style` | STRING | Estilo da cerveja. | Categoria informada na fonte. |
| `review_palate` | DOUBLE | Nota atribuída à sensação no paladar. | Escala de avaliação da fonte. |
| `review_taste` | DOUBLE | Nota atribuída ao sabor. | Escala de avaliação da fonte. |
| `beer_name` | STRING | Nome da cerveja. | Texto original. |
| `beer_abv` | DOUBLE | Teor alcoólico da cerveja, em percentual. | Valor original; admite nulos. |
| `beer_beerid` | INT | Identificador da cerveja. | Código numérico da fonte; admite nulos. |

**Linhagem:** os dados foram ingeridos diretamente do arquivo CSV armazenado no volume gerenciado pelo Unity Catalog. A tabela Bronze constitui a origem das transformações realizadas na Silver e, posteriormente, na Gold.

**Qualidade:** nesta camada, os problemas identificados foram preservados para análise. O diagnóstico incluiu valores ausentes, duplicidades, notas inválidas, características temporais e valores extremos de teor alcoólico. As decisões de tratamento estão documentadas na seção 4 do notebook.

### 6.4.2. Camada Silver

**Tabela:** `workspace.mvp_cervejas.silver_beer_reviews`  
**Camada:** Silver  
**Granularidade:** uma linha por avaliação preservada após a aplicação das regras de qualidade.  
**Origem:** `workspace.mvp_cervejas.bronze_beer_reviews`.

A tabela Silver contém os registros tratados a partir da camada Bronze, preservando os atributos originais das avaliações e acrescentando campos temporais para facilitar as análises.

Nesta etapa, foram excluídos os registros sem informações essenciais de avaliação, tratados os valores inválidos identificados nas notas e convertidos os timestamps Unix para data e horário. Os valores ausentes que não puderam ser preenchidos com segurança foram preservados.

A consulta a seguir apresenta os nomes e tipos das colunas registradas no Unity Catalog, que servirão de referência para a elaboração do dicionário de dados da camada Silver.

In [0]:
%sql
DESCRIBE TABLE workspace.mvp_cervejas.silver_beer_reviews;

col_name,data_type,comment
index,int,null
brewery_id,int,null
brewery_name,string,null
review_time,int,null
review_overall,double,null
review_aroma,double,null
review_appearance,double,null
review_profilename,string,null
beer_style,string,null
review_palate,double,null


#### 6.4.2.1. Dicionário de dados da Silver

A tabela `workspace.mvp_cervejas.silver_beer_reviews` contém 1.047.921 registros e 16 colunas, após a aplicação das regras de qualidade sobre a camada Bronze.

| Coluna | Tipo | Descrição | Domínio e regras |
|---|---|---|---|
| `index` | INT | Identificador da linha na base original. | Preservado para identificar cada avaliação. |
| `brewery_id` | INT | Identificador da cervejaria. | Preservado da Bronze. |
| `brewery_name` | STRING | Nome da cervejaria. | Texto original; admite valores ausentes. |
| `review_time` | INT | Data e horário da avaliação em timestamp Unix. | Sem valores nulos após o tratamento. |
| `review_overall` | DOUBLE | Nota geral atribuída à cerveja. | Sete notas inválidas foram convertidas em nulos. |
| `review_aroma` | DOUBLE | Nota atribuída ao aroma. | Escala de avaliação da fonte. |
| `review_appearance` | DOUBLE | Nota atribuída à aparência. | Sete notas inválidas foram convertidas em nulos. |
| `review_profilename` | STRING | Nome de perfil do autor da avaliação. | Texto proveniente da fonte original. |
| `beer_style` | STRING | Estilo da cerveja. | Categoria textual da fonte original. |
| `review_palate` | DOUBLE | Nota atribuída à sensação no paladar. | Escala de avaliação da fonte. |
| `review_taste` | DOUBLE | Nota atribuída ao sabor. | Escala de avaliação da fonte. |
| `beer_name` | STRING | Nome da cerveja avaliada. | Texto proveniente da fonte original. |
| `beer_abv` | DOUBLE | Teor alcoólico da cerveja, em percentual. | Admite nulos; 42.879 registros sem informação. |
| `beer_beerid` | INT | Identificador da cerveja. | Admite nulos; 163 registros preservados sem identificador. |
| `review_datetime` | TIMESTAMP | Data e horário da avaliação em formato temporal. | Derivado de `review_time`. |
| `review_date` | DATE | Data da avaliação, sem o horário. | Derivada do timestamp original; sem valores nulos. |

**Linhagem:** todos os campos originais foram obtidos da tabela `bronze_beer_reviews`. Os campos `review_datetime` e `review_date` foram acrescentados durante o tratamento, a partir de `review_time`.

**Transformações e qualidade:** foram excluídos 654 registros sem informações essenciais de avaliação. As sete ocorrências de notas inválidas nos campos `review_overall` e `review_appearance` foram substituídas por valores nulos. Os valores ausentes de teor alcoólico e os identificadores de cerveja não recuperáveis foram preservados, evitando a introdução de informações artificiais.

**Resultado:** a camada Silver contém 1.047.921 avaliações tratadas, utilizadas como fonte para a construção das dimensões e da tabela fato da camada Gold.

### 6.4.3. Dimensão de cervejarias

**Tabela:** `workspace.mvp_cervejas.gold_dim_cervejaria`  
**Camada:** Gold  
**Granularidade:** uma linha por cervejaria identificada.  
**Origem:** `silver_beer_reviews`.

A dimensão de cervejarias armazena os identificadores e nomes das cervejarias presentes no conjunto de dados, permitindo relacioná-las às avaliações registradas na tabela fato.

| Coluna | Tipo | Descrição | Domínio e regras |
|---|---|---|---|
| `brewery_id` | INT | Identificador da cervejaria. | Chave única, sem valores nulos. |
| `brewery_name` | STRING | Nome da cervejaria. | Texto proveniente da base original; admite valores nulos. |

**Linhagem:** a dimensão foi construída a partir dos atributos `brewery_id` e `brewery_name` da camada Silver, preservando uma única linha por identificador de cervejaria.

**Qualidade:** foram identificadas 3.817 cervejarias distintas, sem duplicidade de identificadores. Uma cervejaria possui nome ausente, que foi preservado para manter a integridade referencial com a tabela fato.

### 6.4.4. Dimensão de cervejas

**Tabela:** `workspace.mvp_cervejas.gold_dim_cerveja`  
**Camada:** Gold  
**Granularidade:** uma linha por cerveja identificada.  
**Origem:** `silver_beer_reviews`.

A dimensão de cervejas reúne os atributos descritivos de cada produto identificado no conjunto de dados. Ela permite relacionar as avaliações às cervejas, aos seus estilos e às respectivas cervejarias.

| Coluna | Tipo | Descrição | Domínio e regras |
|---|---|---|---|
| `beer_beerid` | INT | Identificador único da cerveja. | Chave única, sem valores nulos. |
| `beer_name` | STRING | Nome da cerveja. | Texto proveniente da fonte original; sem valores nulos na dimensão. |
| `beer_style` | STRING | Estilo da cerveja. | Categoria textual conforme classificação da fonte; sem valores nulos na dimensão. |
| `beer_abv` | DOUBLE | Teor alcoólico da cerveja, em percentual. | Valor numérico; admite nulos quando não informado na fonte. |
| `brewery_id` | INT | Identificador da cervejaria produtora. | Chave de relacionamento com `gold_dim_cervejaria`; sem valores nulos na dimensão. |

**Linhagem:** a dimensão foi construída a partir dos atributos de identificação e descrição das cervejas presentes na camada Silver. Os registros foram organizados de modo a preservar uma única linha por `beer_beerid`, após a verificação de consistência dos atributos associados a cada identificador.

**Qualidade:** foram identificadas 42.551 cervejas distintas, sem duplicidade de identificadores e sem valores ausentes nos campos de nome, estilo e cervejaria. O campo `beer_abv` apresenta 10.823 valores nulos, preservados por não haver informação suficiente para preenchê-los com segurança.

### 6.4.5. Dimensão de tempo

**Tabela:** `workspace.mvp_cervejas.gold_dim_tempo`  
**Camada:** Gold  
**Granularidade:** uma linha por data distinta de avaliação.  
**Origem:** `silver_beer_reviews`.

A dimensão de tempo organiza os atributos temporais das avaliações, permitindo análises por dia, mês, trimestre e ano.

| Coluna | Tipo | Descrição | Domínio e regras |
|---|---|---|---|
| `review_date` | DATE | Data da avaliação. | Chave única, sem valores nulos. |
| `ano` | INT | Ano da avaliação. | Ano extraído de `review_date`. |
| `mes` | INT | Mês da avaliação. | Valores de 1 a 12. |
| `trimestre` | INT | Trimestre da avaliação. | Valores de 1 a 4. |
| `dia` | INT | Dia do mês da avaliação. | Valores de 1 a 31, conforme o mês. |

**Linhagem:** a dimensão foi construída a partir de `review_date`, campo gerado na Silver mediante conversão do timestamp Unix original. Os atributos de ano, mês, trimestre e dia foram derivados dessa data.

**Qualidade:** foram identificadas 4.010 datas distintas, sem duplicidade de chaves e sem valores nulos. A dimensão contempla as datas efetivamente presentes no conjunto de avaliações, e não um calendário contínuo.

### 6.4.6. Tabela fato de avaliações

A consulta a seguir apresenta a estrutura da tabela fato persistida na camada Gold, permitindo conferir os nomes e os tipos de dados de suas 11 colunas.

In [0]:

%sql
DESCRIBE TABLE workspace.mvp_cervejas.gold_fato_avaliacoes;

col_name,data_type,comment
avaliacao_id,int,null
beer_beerid,int,null
brewery_id,int,null
review_date,date,null
review_datetime,timestamp,null
review_profilename,string,null
review_overall,double,null
review_aroma,double,null
review_appearance,double,null
review_palate,double,null


**Tabela:** `workspace.mvp_cervejas.gold_fato_avaliacoes`  
**Camada:** Gold  
**Granularidade:** uma linha por avaliação preservada após o tratamento da camada Silver.  
**Origem:** `silver_beer_reviews`.

A tabela fato armazena as avaliações realizadas pelos usuários, incluindo as notas atribuídas às cervejas e as chaves utilizadas para relacionar os registros às dimensões de cerveja, cervejaria e tempo.

| Coluna | Tipo | Descrição | Domínio e regras |
|---|---|---|---|
| `avaliacao_id` | INT | Identificador único da avaliação, derivado do campo `index` da origem. | Chave única, sem valores nulos. |
| `beer_beerid` | INT | Identificador da cerveja avaliada. | Chave de relacionamento com `gold_dim_cerveja`; admite nulos. |
| `brewery_id` | INT | Identificador da cervejaria. | Chave de relacionamento com `gold_dim_cervejaria`; sem valores nulos na tabela fato. |
| `review_date` | DATE | Data da avaliação. | Relacionamento com `gold_dim_tempo`; sem valores nulos. |
| `review_datetime` | TIMESTAMP | Data e horário da avaliação, convertidos do timestamp Unix original. | Valores temporais correspondentes aos registros preservados. |
| `review_profilename` | STRING | Nome de perfil do usuário responsável pela avaliação. | Texto proveniente da base original. |
| `review_overall` | DOUBLE | Nota geral atribuída à cerveja. | Escala de avaliação da fonte; sete valores inválidos foram convertidos em nulos na Silver. |
| `review_aroma` | DOUBLE | Nota atribuída ao aroma. | Escala de avaliação da fonte. |
| `review_appearance` | DOUBLE | Nota atribuída à aparência. | Escala de avaliação da fonte; sete valores inválidos foram convertidos em nulos na Silver. |
| `review_palate` | DOUBLE | Nota atribuída à sensação da cerveja no paladar. | Escala de avaliação da fonte. |
| `review_taste` | DOUBLE | Nota atribuída ao sabor. | Escala de avaliação da fonte. |

**Linhagem:** os atributos de avaliação e os identificadores foram obtidos da camada Silver. Os campos `review_date` e `review_datetime` resultam da conversão de `review_time`, originalmente armazenado como timestamp Unix. A tabela fato foi construída para permitir consultas integradas com as três dimensões do modelo estrela.

**Integridade referencial:** as verificações realizadas não identificaram chaves preenchidas sem correspondência nas dimensões. Foram preservadas 163 avaliações com `beer_beerid` nulo, pois seus demais atributos de avaliação eram válidos.

## 7. Consultas analíticas na camada Gold

Com o modelo dimensional construído e persistido, serão realizadas consultas analíticas utilizando SQL. As consultas demonstrarão a integração entre a tabela fato e as dimensões, permitindo investigar a distribuição das avaliações por estilo de cerveja, o desempenho das cervejarias e a evolução temporal das avaliações.

### 7.1. Distribuição das avaliações por estilo de cerveja

A primeira análise investiga os estilos de cerveja mais avaliados e suas respectivas notas médias. Para isso, será realizado um relacionamento entre a tabela fato de avaliações e a dimensão de cervejas, utilizando `beer_beerid` como chave.

Os resultados serão ordenados pela quantidade de avaliações, permitindo identificar os estilos mais presentes no conjunto de dados. As avaliações sem identificador de cerveja não serão incluídas nesta consulta, pois não podem ser associadas a um estilo por meio da dimensão.

In [0]:
%sql

SELECT
    c.beer_style AS estilo,
    COUNT(*) AS total_avaliacoes,
    ROUND(AVG(f.review_overall), 2) AS nota_media,
    ROUND(AVG(f.review_taste), 2) AS media_sabor,
    ROUND(AVG(f.review_aroma), 2) AS media_aroma
FROM workspace.mvp_cervejas.gold_fato_avaliacoes AS f
INNER JOIN workspace.mvp_cervejas.gold_dim_cerveja AS c
    ON f.beer_beerid = c.beer_beerid
GROUP BY c.beer_style
ORDER BY total_avaliacoes DESC
LIMIT 20;

estilo,total_avaliacoes,nota_media,media_sabor,media_aroma
American IPA,83616,3.99,3.95,3.92
American Double / Imperial IPA,52934,3.98,4.08,4.09
American Pale Ale (APA),41190,3.84,3.7,3.64
American Double / Imperial Stout,39807,4.05,4.21,4.19
Russian Imperial Stout,34289,4.0,4.13,4.07
American Porter,33946,3.87,3.87,3.81
American Amber / Red Ale,32720,3.82,3.74,3.67
American Strong Ale,24841,3.89,4.06,4.01
Belgian Strong Dark Ale,23123,3.95,4.06,4.0
Fruit / Vegetable Beer,22347,3.49,3.48,3.6


### 7.1.1. Interpretação dos resultados

A consulta identificou a **American IPA** como o estilo com maior quantidade de avaliações na base, totalizando **83.616 registros** e nota média geral de **3,99**. Em seguida, aparecem a American Double / Imperial IPA, com **52.934 avaliações**, e a American Pale Ale (APA), com **41.190**.

Entre os 20 estilos apresentados, a **American Double / Imperial Stout** registrou nota média geral de **4,05**, enquanto a American Adjunct Lager apresentou média de **3,06**.

Os resultados evidenciam que a quantidade de avaliações e a nota média representam indicadores distintos. A elevada frequência de um estilo no conjunto de dados não implica necessariamente uma avaliação média superior.

As conclusões se restringem às avaliações disponíveis na base analisada e não devem ser generalizadas para todos os consumidores de cerveja.

### 7.2. Análise das cervejarias

A segunda consulta analítica investiga a distribuição das avaliações entre as cervejarias presentes na base. Por meio do relacionamento entre a tabela fato e a dimensão de cervejarias, serão calculados o total de avaliações, a quantidade de cervejas distintas avaliadas e a nota média geral de cada fabricante.

Os resultados serão ordenados pelo volume de avaliações, permitindo observar a participação das cervejarias no conjunto de dados e comparar seus indicadores.

In [0]:

%sql

SELECT
    c.brewery_name AS cervejaria,
    COUNT(*) AS total_avaliacoes,
    COUNT(DISTINCT f.beer_beerid) AS cervejas_avaliadas,
    ROUND(AVG(f.review_overall), 2) AS nota_media
FROM workspace.mvp_cervejas.gold_fato_avaliacoes AS f
INNER JOIN workspace.mvp_cervejas.gold_dim_cervejaria AS c
    ON f.brewery_id = c.brewery_id
GROUP BY c.brewery_id, c.brewery_name
ORDER BY total_avaliacoes DESC
LIMIT 20;

cervejaria,total_avaliacoes,cervejas_avaliadas,nota_media
Boston Beer Company (Samuel Adams),39444,133,3.69
Stone Brewing Co.,33066,134,4.04
Sierra Nevada Brewing Co.,28751,140,4.06
"Bell's Brewery, Inc.",25191,121,3.97
Rogue Ales,24083,133,3.91
Founders Brewing Company,20004,147,4.11
Avery Brewing Company,16107,109,3.81
Southern Tier Brewing Company,15868,51,3.88
Great Divide Brewing Company,14935,58,3.97
Goose Island Beer Co.,14534,368,3.99


### 7.2.1. Interpretação dos resultados

A consulta identificou a **Boston Beer Company** como a cervejaria com maior volume de avaliações na base, totalizando **39.444 registros**, distribuídos entre **133 cervejas distintas**, com nota média geral de **3,69**.

Na sequência, aparecem a **Stone Brewing Co.**, com **33.066 avaliações** e média de **4,04**, e a **Sierra Nevada Brewing Co.**, com **28.751 avaliações** e média de **4,06**.

Entre as cervejarias apresentadas, a **Founders Brewing Company** registrou nota média de **4,11**, com **20.004 avaliações**. Já a **Goose Island Beer Co.** apresentou **368 cervejas distintas avaliadas**, evidenciando que a diversidade de produtos e o volume de avaliações são indicadores diferentes.

Os resultados permitem comparar a presença das cervejarias no conjunto de dados e suas notas médias, sem pressupor que maior volume de avaliações represente melhor avaliação ou maior participação no mercado.

### 7.3. Evolução temporal das avaliações

A terceira consulta analítica investiga a evolução anual do volume de avaliações e da nota média geral. Para isso, a tabela fato será relacionada à dimensão de tempo por meio da coluna `review_date`.

O agrupamento por ano permitirá identificar variações na quantidade de avaliações ao longo do período abrangido pela base, bem como observar possíveis mudanças nas notas médias. Os resultados devem ser interpretados considerando que o conjunto de dados representa uma amostra histórica das avaliações disponíveis, e não necessariamente a totalidade da atividade da plataforma.

In [0]:

%sql

SELECT
    t.ano,
    COUNT(*) AS total_avaliacoes,
    ROUND(AVG(f.review_overall), 2) AS nota_media,
    COUNT(DISTINCT f.beer_beerid) AS cervejas_avaliadas
FROM workspace.mvp_cervejas.gold_fato_avaliacoes AS f
INNER JOIN workspace.mvp_cervejas.gold_dim_tempo AS t
    ON f.review_date = t.review_date
GROUP BY t.ano
ORDER BY t.ano;

ano,total_avaliacoes,nota_media,cervejas_avaliadas
1996,1,4.0,1
1998,78,3.89,77
1999,53,3.99,52
2000,60,4.19,58
2001,1203,3.91,643
2002,14584,3.8,3241
2003,35039,3.75,5904
2004,43485,3.79,6840
2005,56690,3.81,7980
2006,82650,3.8,9445


### 7.3.1. Interpretação dos resultados

A análise temporal identificou crescimento expressivo no volume de avaliações a partir de 2002, quando foram registrados **14.584 comentários**, chegando a **226.225 avaliações em 2011**. Nesse mesmo ano, foram avaliadas **19.360 cervejas distintas**, demonstrando a ampliação da diversidade de produtos presentes nos registros.

Apesar do crescimento da atividade, a nota média geral permaneceu relativamente estável durante os anos de maior volume, oscilando aproximadamente entre **3,75 e 3,87** de 2003 a 2011.

O ano de **1996 apresenta apenas uma avaliação**, enquanto não há registros de 1997. Já 2012 contém somente avaliações até janeiro, razão pela qual seu volume reduzido não deve ser interpretado como uma queda anual de atividade.

Os resultados demonstram a aplicação da dimensão de tempo na análise histórica das avaliações. Entretanto, como a base utilizada corresponde a um recorte do conjunto original, as variações observadas não representam necessariamente a evolução completa da atividade da plataforma.

### 7.4. Diversidade de estilos por cervejaria

Esta análise complementa a investigação das cervejarias, calculando a quantidade de cervejas e estilos distintos associados a cada fabricante. O objetivo é identificar a diversidade de produtos presentes na base, utilizando o relacionamento entre as dimensões de cervejas e cervejarias.

Os resultados serão apresentados para as cervejarias com maior quantidade de cervejas distintas avaliadas, sem interpretar a diversidade observada como representativa de todo o portfólio comercial de cada fabricante.

In [0]:

%sql

SELECT
    cj.brewery_name AS cervejaria,
    COUNT(DISTINCT c.beer_beerid) AS cervejas_distintas,
    COUNT(DISTINCT c.beer_style) AS estilos_distintos
FROM workspace.mvp_cervejas.gold_dim_cerveja AS c
INNER JOIN workspace.mvp_cervejas.gold_dim_cervejaria AS cj
    ON c.brewery_id = cj.brewery_id
GROUP BY cj.brewery_id, cj.brewery_name
ORDER BY cervejas_distintas DESC
LIMIT 20;

cervejaria,cervejas_distintas,estilos_distintos
John Harvard's Brewery & Ale House,394,83
Goose Island Beer Co.,368,67
Iron Hill Brewery & Restaurant,316,83
Minneapolis Town Hall Brewery,312,65
Port Brewing Company / Pizza Port,283,59
Cigar City Brewing,201,45
Mikkeller ApS,185,40
Kuhnhenn Brewing Company,177,66
Flossmoor Station Restaurant & Brewery,158,56
Beer Works,157,67


### 7.4.1. Interpretação dos resultados

A consulta identificou a **John Harvard's Brewery & Ale House** com a maior quantidade de cervejas distintas entre as cervejarias apresentadas, totalizando **394 produtos distribuídos em 83 estilos**. Em seguida, aparecem a **Goose Island Beer Co.**, com **368 cervejas e 67 estilos**, e a **Iron Hill Brewery & Restaurant**, com **316 cervejas e 83 estilos**.

Observa-se que a quantidade de cervejas distintas não corresponde necessariamente a uma maior diversidade de estilos. A John Harvard's e a Iron Hill, por exemplo, apresentam a mesma quantidade de estilos identificados, apesar da diferença no número de cervejas.

Os resultados complementam a análise anterior sobre volume de avaliações, permitindo distinguir três indicadores: quantidade de avaliações, diversidade de cervejas e diversidade de estilos. Os valores representam exclusivamente os produtos presentes na base analisada, não os portfólios comerciais completos das cervejarias.

### 7.5. Consistência das avaliações por cerveja

Esta análise investiga a consistência das avaliações individuais das cervejas, considerando três indicadores: quantidade de avaliações, nota média geral e desvio-padrão das notas.

O desvio-padrão será utilizado como medida de dispersão, permitindo observar o grau de concordância entre as avaliações registradas. Valores menores indicam notas mais próximas da média, enquanto valores maiores indicam maior variabilidade.

Para reduzir a influência de produtos com poucas avaliações, serão consideradas apenas cervejas com pelo menos **100 avaliações**. Esse limite constitui um critério exploratório adotado para a presente análise e não representa uma garantia de significância estatística.

In [0]:

%sql

SELECT
    c.beer_name AS cerveja,
    cj.brewery_name AS cervejaria,
    c.beer_style AS estilo,
    COUNT(*) AS total_avaliacoes,
    ROUND(AVG(f.review_overall), 2) AS nota_media,
    ROUND(STDDEV_SAMP(f.review_overall), 2) AS desvio_padrao
FROM workspace.mvp_cervejas.gold_fato_avaliacoes AS f
INNER JOIN workspace.mvp_cervejas.gold_dim_cerveja AS c
    ON f.beer_beerid = c.beer_beerid
INNER JOIN workspace.mvp_cervejas.gold_dim_cervejaria AS cj
    ON f.brewery_id = cj.brewery_id
WHERE f.review_overall IS NOT NULL
GROUP BY
    c.beer_beerid,
    c.beer_name,
    cj.brewery_name,
    c.beer_style
HAVING COUNT(*) >= 100
ORDER BY
    desvio_padrao ASC,
    total_avaliacoes DESC
LIMIT 20;

cerveja,cervejaria,estilo,total_avaliacoes,nota_media,desvio_padrao
Lunch,Maine Beer Company,American IPA,183,4.33,0.35
Heady Topper,The Alchemist,American Double / Imperial IPA,469,4.63,0.36
Thumb Print IIPA,New Glarus Brewing Company,American Double / Imperial IPA,103,4.16,0.36
TrÃ¶egs Perpetual IPA,TrÃ¶egs Brewing Company,American IPA,264,4.12,0.37
Hopworks IPA,Hopworks Urban Brewery,American IPA,114,4.21,0.37
Citra DIPA,Kern River Brewing Company,American Double / Imperial IPA,252,4.63,0.39
Tripel Perfection,Brewery Ommegang,Tripel,142,4.08,0.39
Best Of Beer Camp: Weizenbock - Beer Camp #37,Sierra Nevada Brewing Co.,Weizenbock,285,4.02,0.4
Latitude 48 Deconstructed IPA - Zeus,Boston Beer Company (Samuel Adams),American IPA,161,3.71,0.4
Cantillon Lou Pepe - Gueuze,Brasserie Cantillon,Gueuze,128,4.41,0.4


### 7.5.1. Interpretação dos resultados

A consulta identificou as 20 cervejas com menor desvio-padrão das notas gerais, considerando apenas produtos com pelo menos **100 avaliações**.

A **Lunch**, da Maine Beer Company, apresentou o menor desvio-padrão entre os resultados exibidos (**0,35**), com **183 avaliações** e nota média de **4,33**. A **Heady Topper**, da The Alchemist, apresentou **469 avaliações**, nota média de **4,63** e desvio-padrão de **0,36**, reunindo uma quantidade maior de avaliações com baixa dispersão e média elevada.

Os resultados mostram que a consistência e a avaliação média são indicadores complementares: um desvio-padrão baixo indica maior proximidade entre as notas, mas não significa, isoladamente, que uma cerveja recebeu avaliações positivas.

O recorte de pelo menos 100 avaliações reduz a influência de produtos com poucas observações. As conclusões se restringem às avaliações disponíveis na base analisada.

In [0]:
%sql

SELECT
    c.beer_name AS cerveja,
    cj.brewery_name AS cervejaria,
    c.beer_style AS estilo,
    COUNT(*) AS total_avaliacoes,
    ROUND(AVG(f.review_overall), 2) AS nota_media,
    ROUND(STDDEV_SAMP(f.review_overall), 2) AS desvio_padrao
FROM workspace.mvp_cervejas.gold_fato_avaliacoes AS f
INNER JOIN workspace.mvp_cervejas.gold_dim_cerveja AS c
    ON f.beer_beerid = c.beer_beerid
INNER JOIN workspace.mvp_cervejas.gold_dim_cervejaria AS cj
    ON f.brewery_id = cj.brewery_id
WHERE f.review_overall IS NOT NULL
GROUP BY
    c.beer_beerid,
    c.beer_name,
    cj.brewery_name,
    c.beer_style
HAVING COUNT(*) >= 100
   AND AVG(f.review_overall) >= 4
ORDER BY
    desvio_padrao ASC,
    total_avaliacoes DESC
LIMIT 20;

cerveja,cervejaria,estilo,total_avaliacoes,nota_media,desvio_padrao
Lunch,Maine Beer Company,American IPA,183,4.33,0.35
Heady Topper,The Alchemist,American Double / Imperial IPA,469,4.63,0.36
Thumb Print IIPA,New Glarus Brewing Company,American Double / Imperial IPA,103,4.16,0.36
TrÃ¶egs Perpetual IPA,TrÃ¶egs Brewing Company,American IPA,264,4.12,0.37
Hopworks IPA,Hopworks Urban Brewery,American IPA,114,4.21,0.37
Citra DIPA,Kern River Brewing Company,American Double / Imperial IPA,252,4.63,0.39
Tripel Perfection,Brewery Ommegang,Tripel,142,4.08,0.39
Best Of Beer Camp: Weizenbock - Beer Camp #37,Sierra Nevada Brewing Co.,Weizenbock,285,4.02,0.4
Cantillon Lou Pepe - Gueuze,Brasserie Cantillon,Gueuze,128,4.41,0.4
Exit 16,Flying Fish Brewing Company,American Double / Imperial IPA,120,4.26,0.4


### 7.5.2. Identificação de cervejas com avaliações positivas e consistentes

Para complementar a análise de dispersão, foram selecionadas cervejas com **pelo menos 100 avaliações e nota média igual ou superior a 4**, ordenadas pelo menor desvio-padrão das notas gerais.

Entre os resultados, a **Lunch**, da Maine Beer Company, apresentou o menor desvio-padrão (**0,35**), com **183 avaliações** e nota média de **4,33**. A **Heady Topper**, da The Alchemist, destacou-se pela combinação de **469 avaliações**, nota média de **4,63** e desvio-padrão de **0,36**.

A consulta também identificou outras cervejas com médias elevadas e baixa dispersão, como a **Citra DIPA**, com nota média de **4,63** e desvio-padrão de **0,39**.

Os resultados indicam que é possível combinar a nota média, o volume de avaliações e a dispersão para identificar produtos com avaliações positivas e relativamente consistentes. Entretanto, a baixa variabilidade não comprova, isoladamente, a qualidade de uma cerveja, e as conclusões estão limitadas às avaliações presentes no conjunto de dados.

## 8. Consolidação das análises e resultados

Após a construção e validação da camada Gold, foram realizadas consultas SQL sobre o modelo dimensional para investigar o comportamento das avaliações de cervejas presentes na base BeerAdvocate.

As consultas utilizam a tabela fato de avaliações e as dimensões de cerveja, cervejaria e tempo, permitindo analisar diferentes perspectivas dos dados, como popularidade dos estilos, participação das cervejarias, evolução temporal, diversidade de produtos e consistência das avaliações.

Nesta seção, serão consolidados os resultados obtidos, relacionando as consultas executadas às questões analíticas do MVP. As conclusões consideram exclusivamente os registros disponíveis no conjunto de dados utilizado.

### 8.1. Síntese dos resultados

As consultas realizadas na camada Gold permitiram analisar o conjunto de dados sob cinco perspectivas: popularidade dos estilos, volume de avaliações por cervejaria, evolução temporal, diversidade de produtos e consistência das avaliações.

| Eixo analítico | Principais resultados |
|---|---|
| Popularidade dos estilos | O estilo American IPA apresentou o maior volume entre os estilos analisados, com 83.616 avaliações. |
| Avaliações por cervejaria | A Boston Beer Company registrou 39.444 avaliações, seguida pela Stone Brewing Co., com 33.066. |
| Evolução temporal | O volume de avaliações cresceu expressivamente a partir de 2002, alcançando 226.225 registros em 2011. |
| Diversidade de produtos | A John Harvard's Brewery & Ale House apresentou 394 cervejas distintas, distribuídas por 83 estilos. |
| Consistência das avaliações | A Lunch apresentou o menor desvio-padrão (0,35) entre os produtos selecionados. A Heady Topper reuniu 469 avaliações, nota média de 4,63 e desvio-padrão de 0,36. |

Os resultados demonstram a utilização do modelo dimensional para responder a diferentes questões analíticas por meio de consultas SQL, combinando informações da tabela fato com as dimensões de cerveja, cervejaria e tempo.

As análises são descritivas e se limitam ao conjunto de dados utilizado no MVP. Os volumes de avaliações não representam necessariamente a participação dos produtos ou das cervejarias no mercado cervejeiro.

### 8.2. Limitações das análises

Os resultados apresentados estão restritos ao conjunto de dados utilizado no MVP, que contém avaliações registradas entre 1996 e 2012. Portanto, os volumes e indicadores obtidos não devem ser interpretados como representativos de todo o histórico da plataforma BeerAdvocate ou do mercado cervejeiro.

Durante o tratamento dos dados, foram excluídos da camada Silver 654 registros sem informações essenciais de avaliação. Outros 163 registros sem identificador da cerveja foram preservados na tabela fato, mas não participam das consultas que exigem relacionamento com a dimensão de cervejas.

Os valores ausentes de teor alcoólico foram mantidos como nulos, sem imputação artificial. Além disso, as análises de consistência adotaram um mínimo de 100 avaliações por produto para reduzir a influência de amostras muito pequenas.

Essas decisões foram documentadas ao longo do pipeline para garantir transparência, rastreabilidade e reprodutibilidade dos resultados.

# 9. Arquitetura de dados e governança

Após a implementação do pipeline de dados e das análises, esta seção apresenta a arquitetura adotada no MVP e as práticas utilizadas para organização, qualidade e governança dos dados.

A solução foi desenvolvida no Databricks, utilizando uma arquitetura em camadas para separar os dados conforme seu estágio de processamento. O fluxo foi estruturado nas camadas Bronze, Silver e Gold, permitindo preservar os dados de origem, aplicar tratamentos de qualidade de forma rastreável e disponibilizar estruturas adequadas ao consumo analítico.

Além da organização física e lógica dos dados, foram adotadas práticas relacionadas à qualidade, integridade, rastreabilidade e documentação das transformações realizadas ao longo do pipeline.

## 9.1. Arquitetura Data Lakehouse e organização das camadas

O MVP foi estruturado segundo uma arquitetura em camadas, separando a ingestão, o tratamento e o consumo analítico dos dados.

O arquivo CSV contendo as avaliações de cervejas foi inicialmente armazenado em um volume gerenciado pelo Unity Catalog e posteriormente processado com Apache Spark.

A partir dessa fonte, o pipeline foi organizado nas seguintes camadas:

- **Bronze:** responsável pela preservação dos dados ingeridos com estrutura próxima à fonte original. Os registros foram persistidos na tabela `workspace.mvp_cervejas.bronze_beer_reviews`, sem aplicação das regras de tratamento posteriormente definidas.

- **Silver:** responsável pela aplicação das regras de qualidade identificadas durante o diagnóstico da camada Bronze. Nessa etapa foram tratados registros incompletos e notas inválidas, realizada a conversão do timestamp das avaliações e preservados valores ausentes quando não havia justificativa para imputação.

- **Gold:** responsável pela organização dos dados para consumo analítico. A partir da camada Silver foi construído um modelo dimensional composto pelas dimensões de cerveja, cervejaria e tempo, além da tabela fato de avaliações.

As tabelas processadas foram persistidas em formato Delta e registradas no Unity Catalog. Essa organização permite separar os diferentes estágios do ciclo de vida dos dados e manter a rastreabilidade entre os dados de origem, os dados tratados e as estruturas utilizadas nas análises.

## 9.2. Gestão e Governança de Dados

A gestão e a governança dos dados no MVP foram abordadas por meio da organização dos ativos no Unity Catalog, da definição de regras de qualidade, da documentação das transformações e da validação da integridade dos dados disponibilizados para análise.

### 9.2.1. Catalogação e organização dos dados

Os arquivos e as tabelas do projeto foram organizados no catálogo `workspace`, dentro do schema `mvp_cervejas`. O arquivo CSV original foi armazenado no volume gerenciado `beer_reviews_raw`, enquanto as tabelas Bronze, Silver e Gold foram registradas no Unity Catalog.

Essa estrutura permite identificar os ativos utilizados pelo projeto e separar os dados conforme sua finalidade. A utilização de nomes padronizados para as tabelas também facilita a identificação de sua posição no pipeline.

O Unity Catalog oferece recursos de gerenciamento de permissões, metadados e linhagem de dados. Neste MVP, sua utilização foi demonstrada principalmente pela organização e pelo registro dos ativos de dados. Não foram implementadas políticas específicas de controle de acesso ou auditoria.

### 9.2.2. Qualidade e integridade dos dados

A qualidade dos dados foi avaliada inicialmente na camada Bronze, com a identificação de valores ausentes, duplicidades, notas inválidas, inconsistências de identificação e características dos registros temporais.

Com base nesse diagnóstico, foram definidas e documentadas regras de tratamento para a camada Silver. Entre as principais decisões, destacam-se:

- Exclusão de 654 registros sem informações essenciais de avaliação;
- Conversão dos timestamps Unix para datas e horários interpretáveis;
- Substituição de sete ocorrências de notas inválidas por valores nulos nos campos afetados;
- Preservação dos valores ausentes de teor alcoólico, evitando imputações sem evidência suficiente;
- Manutenção dos registros sem identificador da cerveja quando continham informações válidas de avaliação.

Na camada Gold, foram realizadas verificações de unicidade das chaves das dimensões e de integridade referencial da tabela fato. As verificações não identificaram referências inválidas entre as chaves preenchidas da tabela fato e suas respectivas dimensões.

### 9.2.3. Rastreabilidade e documentação

A preservação dos dados originais na camada Bronze, juntamente com a documentação das regras aplicadas na Silver e das transformações realizadas na Gold, permite compreender o caminho percorrido pelos dados até as consultas analíticas.

As etapas do pipeline foram registradas no notebook, incluindo os diagnósticos, os critérios de tratamento, a construção do modelo dimensional e as validações realizadas. Dessa forma, os resultados apresentados podem ser relacionados às decisões adotadas durante o processamento.

A documentação das transformações constitui uma prática de rastreabilidade do projeto, embora não substitua uma solução completa de auditoria e monitoramento automatizado.

# 10. Conclusão e autoavaliação

## 10.1. Conclusão

O presente MVP teve como objetivo desenvolver um pipeline de Engenharia de Dados na plataforma Databricks, utilizando avaliações de cervejas como fonte para a implementação de uma arquitetura Data Lakehouse e de um modelo dimensional voltado à análise de dados.

A solução foi estruturada nas camadas Bronze, Silver e Gold. A camada Bronze preservou os 1.048.575 registros ingeridos do arquivo CSV original. Na camada Silver, foram aplicadas as regras de qualidade definidas após o diagnóstico dos dados, resultando em 1.047.921 avaliações tratadas. Por fim, a camada Gold disponibilizou um modelo dimensional composto por três dimensões — cervejas, cervejarias e tempo — e uma tabela fato de avaliações.

Durante o desenvolvimento, foram identificados e tratados registros incompletos e notas inválidas, convertidos os registros temporais e preservados valores ausentes quando não havia justificativa suficiente para seu preenchimento. Também foram realizadas verificações de unicidade das dimensões e de integridade referencial da tabela fato.

As consultas SQL realizadas sobre a camada Gold permitiram explorar cinco perspectivas analíticas: popularidade dos estilos de cerveja, volume de avaliações por cervejaria, evolução temporal das avaliações, diversidade de produtos e consistência das notas atribuídas às cervejas. Os resultados demonstraram a utilização do modelo dimensional para integrar diferentes atributos e responder a perguntas analíticas.

As tabelas foram persistidas em formato Delta e organizadas no Unity Catalog, enquanto as transformações e decisões de qualidade foram documentadas no notebook. Dessa forma, o projeto reuniu etapas de ingestão, processamento, modelagem, validação e consumo analítico em um mesmo ambiente.

Os resultados devem ser interpretados dentro dos limites do conjunto de dados utilizado, sem generalizações para todo o histórico da plataforma de origem ou para o mercado cervejeiro.

## 10.2. Autoavaliação

O desenvolvimento do MVP permitiu aplicar, de forma integrada, conceitos de Engenharia de Dados relacionados à ingestão, ao processamento, à qualidade, à modelagem dimensional e à organização dos dados para análise.

Um dos principais desafios técnicos foi investigar as inconsistências presentes na base original e definir critérios de tratamento que preservassem o máximo possível de informações válidas. A identificação de registros incompletos, notas inválidas e valores ausentes de teor alcoólico exigiu a avaliação de diferentes alternativas antes da implementação das regras na camada Silver.

A construção do modelo dimensional também exigiu atenção à granularidade das tabelas, à unicidade das chaves e à integridade dos relacionamentos entre a tabela fato e as dimensões. As validações realizadas contribuíram para verificar se as transformações preservaram a consistência necessária às consultas analíticas.

A utilização do Databricks possibilitou exercitar o processamento de dados com Apache Spark, a persistência em formato Delta, a organização dos ativos no Unity Catalog e a execução de consultas SQL sobre as tabelas da camada Gold.

Como possibilidades de evolução, destacam-se a automatização do pipeline, a implementação de verificações recorrentes de qualidade, a definição de permissões de leitura e escrita por perfil de usuário no Unity Catalog e o desenvolvimento de painéis interativos para visualização dos resultados.

O projeto contribuiu para consolidar a compreensão do fluxo de Engenharia de Dados, desde a avaliação da fonte original até a disponibilização de informações estruturadas para análise, evidenciando a importância da qualidade, da documentação e da rastreabilidade ao longo de todas as etapas.